# Tool-call verifier / ranker training notebook, production edition

This notebook trains a small model sidecar for tool-call guardrails. It is meant to classify or rank candidate tool calls after deterministic checks already handled syntax, schema, unknown tool names, required-step enforcement, and terminal-tool rules.

Main changes in this production edition:

- defaults to `LABEL_MODE = "production"`
- keeps Hugging Face upload enabled by default, private by default
- adds GPU profiles for T4, L4/A100, and H100-class Colab runtimes
- uses group-safe train/validation/test splits to reduce leakage from generated hard negatives
- includes a versioned input schema, artifact manifest, thresholds, and serializer fixtures
- adds early stopping because macro F1 can peak before cross-entropy loss finishes improving
- exports ONNX and optional dynamic quantization for Rust-side inference
- includes a Rust implementation appendix at the end

## Read your latest run correctly

Your latest T4 run reached roughly:

- epoch 1 validation macro F1: `0.4822`
- epoch 2 validation macro F1: `0.4464`
- validation loss improved from `0.8487` to `0.7739`, but macro F1 dropped

That means the model was optimizing cross-entropy but not the metric you care about. This notebook uses `macro_f1` for best-checkpoint selection and adds early stopping. More epochs are allowed, but only the best macro-F1 checkpoint is kept.

In [ ]:
#@title Install dependencies
!pip -q install -U \
  "datasets>=2.20.0" \
  "transformers>=4.41.0" \
  "accelerate>=0.30.0" \
  "evaluate>=0.4.2" \
  "scikit-learn>=1.3.0" \
  "huggingface_hub>=0.23.0" \
  "jsonschema>=4.22.0" \
  "optimum[onnxruntime]>=1.20.0" \
  "onnxruntime>=1.18.0" \
  "onnx>=1.16.0" \
  "sentencepiece>=0.2.0"

In [ ]:
#@title Imports, paths, and GPU/profile selector
import os
import re
import gc
import ast
import json
import time
import math
import random
import shutil
import hashlib
import glob
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, load_dataset, Value
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit, train_test_split

import torch
from torch import nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/content/toolcall-verifier")
DATA_DIR = WORKDIR / "data"
MODEL_DIR = WORKDIR / "model"
ONNX_DIR = WORKDIR / "onnx"
ARTIFACT_DIR = WORKDIR / "artifacts"
for p in [WORKDIR, DATA_DIR, MODEL_DIR, ONNX_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TOOLCALL_INPUT_SCHEMA_VERSION_V1 = "toolcall-verifier-input/v1"
TOOLCALL_INPUT_SCHEMA_VERSION_V2 = "toolcall-verifier-input/v2"
INPUT_SCHEMA_VERSION = TOOLCALL_INPUT_SCHEMA_VERSION_V1
ARTIFACT_SCHEMA_VERSION = "toolcall-verifier-artifact/v1"
FINAL_RESPONSE_ARTIFACT_SCHEMA_VERSION = "final-response-verifier-artifact/v1"
FINAL_RESPONSE_INPUT_SCHEMA_VERSION = "final-response-verifier-input/v1"
FINAL_RESPONSE_THRESHOLDS_SCHEMA_VERSION = "final-response-verifier-thresholds/v1"
FINAL_RESPONSE_SERIALIZER_VERSION = "serialize_final_response_state_v1"
USE_SERIALIZER_V2 = False  #@param {type:"boolean"}
SERIALIZER_VERSION = "serialize_state_v2" if USE_SERIALIZER_V2 else "serialize_state_v1"
if USE_SERIALIZER_V2:
    INPUT_SCHEMA_VERSION = TOOLCALL_INPUT_SCHEMA_VERSION_V2

BASE_MODEL = "microsoft/deberta-v3-small"  #@param {type:"string"}

# Pick "auto" for normal use. On T4, auto selects the proven 4-epoch baseline.
GPU_PROFILE = "auto"  #@param ["auto", "debug_smoke", "t4_fast", "t4_proven", "l4_balanced", "a100_40gb", "high_vram_fast", "high_vram_quality", "high_vram_full_context"]
LABEL_MODE = "production"  #@param ["production", "diagnostic"]

# Optional manual overrides. Leave 0 / 0.0 to use the selected GPU profile.
MAX_PER_SOURCE_OVERRIDE = 0  #@param {type:"integer"}
MAX_LENGTH_OVERRIDE = 0  #@param {type:"integer"}
NUM_EPOCHS_OVERRIDE = 0  #@param {type:"integer"}
TRAIN_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
EVAL_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
GRAD_ACCUM_OVERRIDE = 0  #@param {type:"integer"}
LEARNING_RATE_OVERRIDE = 0.0  #@param {type:"number"}

# Keep class weights off by default. Previous weighted + synthetic rare-class runs regressed badly.
USE_CLASS_WEIGHTS = False  #@param {type:"boolean"}
MAX_CLASS_WEIGHT = 2.0  #@param {type:"number"}

# Keep Forge augmentation off for the main public-data run. Turn on as a separate ablation only.
ENABLE_FORGE_AUGMENTATION = False  #@param {type:"boolean"}
FORGE_AUGMENTATION_PER_LABEL = 100  #@param {type:"integer"}
ENABLE_FINAL_RESPONSE_VERIFIER = False  #@param {type:"boolean"}
FINAL_RESPONSE_MAX_PER_LABEL = 5_000  #@param {type:"integer"}
FINAL_RESPONSE_MAX_LENGTH_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_NUM_EPOCHS_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_TRAIN_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_EVAL_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_GRAD_ACCUM_OVERRIDE = 0  #@param {type:"integer"}

# Hugging Face upload is intentionally enabled by default, but private by default.
# For experimental v2/final-response artifacts, provenance files mark the artifact as shadow-first.
UPLOAD_TO_HUB = True  #@param {type:"boolean"}
HF_DATASET_REPO = "cowWhySo/toolcall-verifier-dataset"  #@param {type:"string"}
HF_MODEL_REPO = "cowWhySo/toolcall-verifier-classifier-production"  #@param {type:"string"}
HF_FINAL_RESPONSE_MODEL_REPO = "cowWhySo/final-response-verifier-classifier-production"  #@param {type:"string"}
PRIVATE = True  #@param {type:"boolean"}

# Baseline from the first completed T4 run. Used only for comparison in summaries.
T4_PROVEN_BASELINE = {
    "profile": "t4_proven",
    "gpu": "Tesla T4",
    "max_length": 768,
    "epochs_completed": 4,
    "best_epoch": 4,
    "best_accuracy": 0.839146,
    "best_macro_f1": 0.741871,
    "best_macro_f1_all_labels": 0.593497,
    "train_runtime_seconds": 5144.4353,
    "steps_per_second": 0.915,
}


def cuda_info() -> Dict[str, Any]:
    if not torch.cuda.is_available():
        return {"available": False, "name": "CPU", "capability": None, "total_gb": 0.0}
    props = torch.cuda.get_device_properties(0)
    return {
        "available": True,
        "name": torch.cuda.get_device_name(0),
        "capability": torch.cuda.get_device_capability(0),
        "total_gb": round(props.total_memory / (1024 ** 3), 1),
    }

GPU_INFO = cuda_info()

PROFILE_CONFIGS = {
    # Fast correctness run. Use this to validate cells, schemas, export, upload, and ONNX parity.
    "debug_smoke": {
        "max_per_source": 500,
        "max_length": 512,
        "epochs": 2,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "grad_accum": 1,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.06,
        "early_stopping_patience": 1,
        "max_tools_in_prompt": 16,
        "dataloader_num_workers": 2,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Cheaper T4 iteration. Use for notebook/debug cycles, not final artifacts.
    "t4_fast": {
        "max_per_source": 3_000,
        "max_length": 640,
        "epochs": 3,
        "train_batch_size": 12,
        "eval_batch_size": 24,
        "grad_accum": 3,
        "learning_rate": 1.2e-5,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 1,
        "max_tools_in_prompt": 20,
        "dataloader_num_workers": 2,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Proven T4 baseline from the first completed run:
    # epoch 4 validation macro_f1=0.741871, accuracy=0.839146, runtime about 1h25m on T4.
    "t4_proven": {
        "max_per_source": 5_000,
        "max_length": 768,
        "epochs": 4,
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "grad_accum": 4,
        "learning_rate": 1e-5,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 24,
        "dataloader_num_workers": 2,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Good profile for Colab L4/A100-40GB style runtimes.
    "l4_balanced": {
        "max_per_source": 10_000,
        "max_length": 1024,
        "epochs": 4,
        "train_batch_size": 24,
        "eval_batch_size": 48,
        "grad_accum": 1,
        "learning_rate": 9e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 32,
        "dataloader_num_workers": 4,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Higher memory profile for A100 40GB. Keeps p95-ish context and larger batches.
    "a100_40gb": {
        "max_per_source": 15_000,
        "max_length": 1024,
        "epochs": 5,
        "train_batch_size": 32,
        "eval_batch_size": 64,
        "grad_accum": 1,
        "learning_rate": 8e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 40,
        "dataloader_num_workers": 6,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # First profile to try on a 80-100GB GPU. Fast enough to iterate while using longer context.
    "high_vram_fast": {
        "max_per_source": 15_000,
        "max_length": 1024,
        "epochs": 4,
        "train_batch_size": 64,
        "eval_batch_size": 128,
        "grad_accum": 1,
        "learning_rate": 8e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 40,
        "dataloader_num_workers": 8,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Recommended quality profile for your ~98GB runtime. 1280 covers close to p99 from your token sample.
    "high_vram_quality": {
        "max_per_source": 40_000,
        "max_length": 1280,
        "epochs": 5,
        "train_batch_size": 64,
        "eval_batch_size": 128,
        "grad_accum": 1,
        "learning_rate": 6e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 48,
        "dataloader_num_workers": 8,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Expensive ablation for very long contexts. Use only after high_vram_quality works.
    "high_vram_full_context": {
        "max_per_source": 40_000,
        "max_length": 1536,
        "epochs": 5,
        "train_batch_size": 48,
        "eval_batch_size": 96,
        "grad_accum": 1,
        "learning_rate": 6e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 56,
        "dataloader_num_workers": 8,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },
}


def choose_auto_profile(info: Dict[str, Any]) -> str:
    if not info["available"]:
        return "debug_smoke"
    name = str(info["name"]).lower()
    gb = float(info["total_gb"])
    if "h100" in name or gb >= 85:
        return "high_vram_quality"
    if gb >= 70:
        return "high_vram_quality"
    if "a100" in name or gb >= 35:
        return "a100_40gb"
    if "l4" in name or gb >= 20:
        return "l4_balanced"
    if "t4" in name or gb >= 14:
        return "t4_proven"
    return "debug_smoke"

SELECTED_PROFILE = choose_auto_profile(GPU_INFO) if GPU_PROFILE == "auto" else GPU_PROFILE
if SELECTED_PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"Unknown GPU_PROFILE={GPU_PROFILE!r}. Valid: {sorted(PROFILE_CONFIGS)}")

CFG = dict(PROFILE_CONFIGS[SELECTED_PROFILE])

# Apply manual overrides last.
if MAX_PER_SOURCE_OVERRIDE > 0:
    CFG["max_per_source"] = int(MAX_PER_SOURCE_OVERRIDE)
if MAX_LENGTH_OVERRIDE > 0:
    CFG["max_length"] = int(MAX_LENGTH_OVERRIDE)
if NUM_EPOCHS_OVERRIDE > 0:
    CFG["epochs"] = int(NUM_EPOCHS_OVERRIDE)
if TRAIN_BATCH_SIZE_OVERRIDE > 0:
    CFG["train_batch_size"] = int(TRAIN_BATCH_SIZE_OVERRIDE)
if EVAL_BATCH_SIZE_OVERRIDE > 0:
    CFG["eval_batch_size"] = int(EVAL_BATCH_SIZE_OVERRIDE)
if GRAD_ACCUM_OVERRIDE > 0:
    CFG["grad_accum"] = int(GRAD_ACCUM_OVERRIDE)
if LEARNING_RATE_OVERRIDE > 0:
    CFG["learning_rate"] = float(LEARNING_RATE_OVERRIDE)

RUN_PROFILE = SELECTED_PROFILE  # kept for artifact manifests/backward compatibility
MAX_PER_SOURCE = CFG["max_per_source"]
MAX_LENGTH = CFG["max_length"]
NUM_EPOCHS = CFG["epochs"]
TRAIN_BATCH_SIZE = CFG["train_batch_size"]
EVAL_BATCH_SIZE = CFG["eval_batch_size"]
GRADIENT_ACCUMULATION_STEPS = CFG["grad_accum"]
LEARNING_RATE = CFG["learning_rate"]
WARMUP_RATIO = CFG["warmup_ratio"]
EARLY_STOPPING_PATIENCE = CFG["early_stopping_patience"]
MAX_TOOLS_IN_PROMPT = CFG["max_tools_in_prompt"]
DATALOADER_NUM_WORKERS = CFG["dataloader_num_workers"]
GRADIENT_CHECKPOINTING = CFG["gradient_checkpointing"]
OPTIMIZER = CFG["optimizer"]

AUTO_FIND_BATCH_SIZE = True
GROUP_BY_LENGTH = True
MAX_PER_LABEL = 50_000
VALID_SIZE = 0.10
TEST_SIZE = 0.10

print("Detected GPU:", json.dumps(GPU_INFO, indent=2))
print("Requested GPU_PROFILE:", GPU_PROFILE)
print("Selected RUN_PROFILE:", RUN_PROFILE)
print("Effective profile config:")
print(json.dumps(CFG, indent=2))
print("Effective train batch:", TRAIN_BATCH_SIZE * max(1, torch.cuda.device_count()), "x grad_accum", GRADIENT_ACCUMULATION_STEPS)
print("T4 proven baseline:", json.dumps(T4_PROVEN_BASELINE, indent=2))


In [ ]:
#@title GPU capability detection and Hugging Face auth

# Reuse GPU_INFO from the profile selector cell.
def cuda_capability():
    if not torch.cuda.is_available():
        return None
    return torch.cuda.get_device_capability(0)

def is_ampere_or_newer() -> bool:
    cap = cuda_capability()
    return cap is not None and cap[0] >= 8

USE_CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if USE_CUDA else "CPU"
CAP = cuda_capability()

# T4 is compute capability 7.5: it supports fp16, but not true bf16 or tf32.
# Ampere+ supports tf32 and usually bf16. Use capability gates in addition to PyTorch feature flags.
USE_TF32 = bool(USE_CUDA and is_ampere_or_newer())
USE_BF16 = bool(USE_CUDA and is_ampere_or_newer() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(USE_CUDA and not USE_BF16)

if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32
    # Faster matmul path on Ampere/Hopper where available. Safe no-op on older runtimes.
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("Device:", GPU_NAME)
print("CUDA capability:", CAP)
print("GPU total VRAM GB:", GPU_INFO.get("total_gb"))
print("Precision flags:", {"fp16": USE_FP16, "bf16": USE_BF16, "tf32": USE_TF32})
print("Training profile:", RUN_PROFILE)

if USE_CUDA:
    try:
        !nvidia-smi
    except Exception:
        pass

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Authenticated to Hugging Face.")
else:
    print("No HF_TOKEN found. Public dataset loading works, but upload will fail unless you set HF_TOKEN.")


## 1. Label schema

Production mode collapses deterministic failures into `deterministic_invalid`. The Rust guardrails should still own deterministic blocks. The classifier should mainly help with semantic ambiguity, nudge specificity, and ranking.

In [ ]:
RAW_LABELS = [
    "valid",
    "wrong_tool_semantic",
    "wrong_arguments_semantic",
    "invalid_args_schema",
    "missing_required_args",
    "unknown_tool",
    "premature_terminal",
    "missing_prerequisite",
    "unsafe_parallel_batch",
    "tool_not_needed",
    "needs_clarification",
    "malformed_tool_call",
]

SEMANTIC_LABEL_MAP = {
    "valid": "valid",
    "wrong_tool_semantic": "wrong_tool_semantic",
    "wrong_arguments_semantic": "wrong_arguments_semantic",
    "tool_not_needed": "tool_not_needed",
    "needs_clarification": "needs_clarification",

    # Deterministic Rust guardrail failures. Keep them as a single non-authoritative bucket.
    "invalid_args_schema": "deterministic_invalid",
    "missing_required_args": "deterministic_invalid",
    "unknown_tool": "deterministic_invalid",
    "premature_terminal": "deterministic_invalid",
    "missing_prerequisite": "deterministic_invalid",
    "unsafe_parallel_batch": "deterministic_invalid",
    "malformed_tool_call": "deterministic_invalid",
}

FINAL_RESPONSE_LABELS = [
    "valid_final_response",
    "missing_tool_fact",
    "contradicts_tool_result",
    "unsupported_claim",
    "failed_to_acknowledge_data_gap",
]
final_response_label2id = {label: i for i, label in enumerate(FINAL_RESPONSE_LABELS)}
final_response_id2label = {i: label for label, i in final_response_label2id.items()}

def normalize_label(label: str) -> str:
    if LABEL_MODE == "production":
        return SEMANTIC_LABEL_MAP.get(label, label)
    return label

if LABEL_MODE == "production":
    LABELS = [
        "valid",
        "wrong_tool_semantic",
        "wrong_arguments_semantic",
        "tool_not_needed",
        "needs_clarification",
        "deterministic_invalid",
    ]
else:
    LABELS = RAW_LABELS

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

labels_doc = {
    "label_mode": LABEL_MODE,
    "raw_labels": RAW_LABELS,
    "labels": LABELS,
    "semantic_label_map": SEMANTIC_LABEL_MAP,
    "label2id": label2id,
    "id2label": id2label,
    "final_response_labels": FINAL_RESPONSE_LABELS,
    "final_response_label2id": final_response_label2id,
    "final_response_id2label": final_response_id2label,
}
(DATA_DIR / "labels.json").write_text(json.dumps(labels_doc, indent=2))
labels_doc


## 2. Canonical input schema and serializer

The Rust side should serialize the same structured context into the same text format before tokenization. Do not train on one format and infer with another.

In [ ]:
BASE_WORKFLOW_STATE_SCHEMA = {
    "type": "object",
    "required": ["required_steps", "completed_steps", "pending_steps", "terminal_tools"],
    "properties": {
        "required_steps": {"type": "array", "items": {"type": "string"}},
        "completed_steps": {"type": "array", "items": {"type": "string"}},
        "pending_steps": {"type": "array", "items": {"type": "string"}},
        "terminal_tools": {"type": "array", "items": {"type": "string"}},
        "recent_errors": {"type": "array", "items": {"type": "string"}},
    },
    "additionalProperties": False,
}

SCORING_METADATA_SCHEMA = {
    "type": "object",
    "properties": {
        "scenario_family": {"type": ["string", "null"]},
        "requires_transform": {"type": ["boolean", "null"]},
        "requires_synthesis": {"type": ["boolean", "null"]},
        "requires_all_tool_facts": {"type": ["boolean", "null"]},
        "must_acknowledge_missing_data": {"type": ["boolean", "null"]},
    },
    "additionalProperties": False,
}

BASE_TOOLCALL_INPUT_PROPERTIES = {
    "schema_version": {"type": "string"},
    "user_request": {"type": "string"},
    "workflow_state": BASE_WORKFLOW_STATE_SCHEMA,
    "available_tools": {
        "type": "array",
        "items": {
            "type": "object",
            "required": ["name", "description", "parameters"],
            "properties": {
                "name": {"type": "string"},
                "description": {"type": "string"},
                "parameters": {"type": "object"},
            },
            "additionalProperties": True,
        },
    },
    "candidate_call": {
        "oneOf": [
            {
                "type": "object",
                "required": ["name", "arguments"],
                "properties": {
                    "name": {"type": "string"},
                    "arguments": {"type": "object"},
                },
                "additionalProperties": True,
            },
            {"type": "array"},
            {"type": "object"},
        ]
    },
}

INPUT_SCHEMA_V1 = {
    "$id": TOOLCALL_INPUT_SCHEMA_VERSION_V1,
    "type": "object",
    "required": ["schema_version", "user_request", "workflow_state", "available_tools", "candidate_call"],
    "properties": dict(BASE_TOOLCALL_INPUT_PROPERTIES, schema_version={"const": TOOLCALL_INPUT_SCHEMA_VERSION_V1}),
    "additionalProperties": False,
}

INPUT_SCHEMA_V2 = {
    "$id": TOOLCALL_INPUT_SCHEMA_VERSION_V2,
    "type": "object",
    "required": ["schema_version", "user_request", "workflow_state", "available_tools", "candidate_call"],
    "properties": dict(
        BASE_TOOLCALL_INPUT_PROPERTIES,
        schema_version={"const": TOOLCALL_INPUT_SCHEMA_VERSION_V2},
        metadata=SCORING_METADATA_SCHEMA,
    ),
    "additionalProperties": False,
}

INPUT_SCHEMA = INPUT_SCHEMA_V2 if USE_SERIALIZER_V2 else INPUT_SCHEMA_V1

FINAL_RESPONSE_INPUT_SCHEMA = {
    "$id": FINAL_RESPONSE_INPUT_SCHEMA_VERSION,
    "type": "object",
    "required": [
        "schema_version", "user_request", "workflow_state", "required_facts",
        "tool_trace", "tool_results", "candidate_final_response",
    ],
    "properties": {
        "schema_version": {"const": FINAL_RESPONSE_INPUT_SCHEMA_VERSION},
        "user_request": {"type": "string"},
        "workflow_state": BASE_WORKFLOW_STATE_SCHEMA,
        "required_facts": {"type": "array", "items": {"type": "string"}},
        "tool_trace": {"type": "array", "items": {"type": "string"}},
        "tool_results": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["tool_name", "content"],
                "properties": {
                    "tool_name": {"type": "string"},
                    "content": {"type": "string"},
                },
                "additionalProperties": True,
            },
        },
        "candidate_final_response": {"type": "string"},
        "metadata": SCORING_METADATA_SCHEMA,
    },
    "additionalProperties": False,
}

(DATA_DIR / "input_schema.json").write_text(json.dumps(INPUT_SCHEMA, indent=2))
(DATA_DIR / "input_schema_v1.json").write_text(json.dumps(INPUT_SCHEMA_V1, indent=2))
(DATA_DIR / "input_schema_v2.json").write_text(json.dumps(INPUT_SCHEMA_V2, indent=2))
(DATA_DIR / "final_response_input_schema.json").write_text(json.dumps(FINAL_RESPONSE_INPUT_SCHEMA, indent=2))
INPUT_SCHEMA


In [ ]:
@dataclass
class VerifierRow:
    id: str
    source: str
    input_text: str
    label: str
    rank_score: float
    metadata: Dict[str, Any]

@dataclass
class FinalResponseVerifierRow:
    id: str
    source: str
    input_text: str
    label: str
    rank_score: float
    metadata: Dict[str, Any]

def stable_id(*parts: Any) -> str:
    h = hashlib.sha256(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:16]
    return h

def compact_json(obj: Any, max_chars: int = 2500) -> str:
    try:
        s = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except TypeError:
        s = str(obj)
    if len(s) > max_chars:
        return s[:max_chars] + "...<truncated>"
    return s

def stable_toolset_hash(tools: List[Dict[str, Any]]) -> str:
    names = sorted([str(t.get("name", "")) for t in tools])
    return stable_id(names)

def normalize_tool_for_prompt(tool: Dict[str, Any]) -> Dict[str, Any]:
    if tool.get("type") == "function" and isinstance(tool.get("function"), dict):
        f = tool["function"]
        return {
            "name": str(f.get("name", "unknown_tool")),
            "description": str(f.get("description", "")),
            "parameters": f.get("parameters") or {"type": "object", "properties": {}},
        }
    return {
        "name": str(tool.get("name", tool.get("tool_name", "unknown_tool"))),
        "description": str(tool.get("description", tool.get("desc", ""))),
        "parameters": tool.get("parameters") or {"type": "object", "properties": {}},
    }

def candidate_tool_names(candidate: Any) -> List[str]:
    if isinstance(candidate, list):
        out = []
        for item in candidate:
            out.extend(candidate_tool_names(item))
        return out
    if isinstance(candidate, dict):
        if "name" in candidate:
            return [str(candidate["name"])]
        if isinstance(candidate.get("function"), dict) and "name" in candidate["function"]:
            return [str(candidate["function"]["name"])]
        if isinstance(candidate.get("tool"), dict):
            return candidate_tool_names(candidate["tool"])
    return []

def select_relevant_tools(
    tools: List[Dict[str, Any]],
    candidate: Any,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    max_tools: int = MAX_TOOLS_IN_PROMPT,
) -> List[Dict[str, Any]]:
    normalized = [normalize_tool_for_prompt(t) for t in tools]
    by_name = {t.get("name"): t for t in normalized}
    priority_names = []
    priority_names.extend(candidate_tool_names(candidate))
    priority_names.extend(pending_steps or [])
    priority_names.extend(terminal_tools or [])

    selected = []
    seen = set()
    for name in priority_names:
        if name in by_name and name not in seen:
            selected.append(by_name[name])
            seen.add(name)
    for t in normalized:
        name = t.get("name")
        if name not in seen:
            selected.append(t)
            seen.add(name)
        if len(selected) >= max_tools:
            break
    return selected[:max_tools]

def serialize_tool(tool: Dict[str, Any]) -> str:
    t = normalize_tool_for_prompt(tool)
    return f"{t['name']}: {t['description']}\nPARAMETERS: {compact_json(t['parameters'], 1200)}"

def infer_scoring_metadata(scenario_family: Optional[str] = None, **overrides: Any) -> Dict[str, Any]:
    family = (scenario_family or "").replace("_stateful", "")
    metadata = {
        "scenario_family": scenario_family,
        "requires_transform": family in {"argument_transformation", "inconsistent_api_recovery"},
        "requires_synthesis": family in {"grounded_synthesis", "data_gap_recovery", "data_gap_recovery_extended"},
        "requires_all_tool_facts": family in {"grounded_synthesis", "data_gap_recovery", "data_gap_recovery_extended", "argument_transformation"},
        "must_acknowledge_missing_data": family in {"data_gap_recovery", "data_gap_recovery_extended"},
    }
    metadata.update({k: v for k, v in overrides.items() if v is not None})
    return metadata

def build_input_object(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
    scoring_metadata: Optional[Dict[str, Any]] = None,
    schema_version: Optional[str] = None,
) -> Dict[str, Any]:
    schema_version = schema_version or INPUT_SCHEMA_VERSION
    relevant_tools = select_relevant_tools(tools, candidate, pending_steps, terminal_tools)
    obj = {
        "schema_version": schema_version,
        "user_request": user_request,
        "workflow_state": {
            "required_steps": required_steps or [],
            "completed_steps": completed_steps or [],
            "pending_steps": pending_steps or [],
            "terminal_tools": terminal_tools or [],
            "recent_errors": recent_errors or [],
        },
        "available_tools": relevant_tools,
        "candidate_call": candidate,
    }
    if schema_version == TOOLCALL_INPUT_SCHEMA_VERSION_V2:
        obj["metadata"] = scoring_metadata or infer_scoring_metadata(None)
    return obj

def serialize_state_v1(input_obj: Dict[str, Any]) -> str:
    ws = input_obj["workflow_state"]
    tool_text = "\n\n".join(serialize_tool(t) for t in input_obj["available_tools"])
    return f"""SCHEMA_VERSION:
{input_obj['schema_version']}

USER_REQUEST:
{input_obj['user_request']}

WORKFLOW_STATE:
required_steps={ws.get('required_steps', [])}
completed_steps={ws.get('completed_steps', [])}
pending_steps={ws.get('pending_steps', [])}
terminal_tools={ws.get('terminal_tools', [])}
recent_errors={ws.get('recent_errors', [])}

AVAILABLE_TOOLS:
{tool_text}

CANDIDATE_CALL:
{compact_json(input_obj['candidate_call'], 2400)}
""".strip()

def _json_or_null(value: Any) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    return json.dumps(value, ensure_ascii=False)

def serialize_state_v2(input_obj: Dict[str, Any]) -> str:
    metadata = input_obj.get("metadata") or {}
    base = serialize_state_v1(input_obj)
    return base + f"""

SCORING_METADATA:
scenario_family={_json_or_null(metadata.get('scenario_family'))}
requires_transform={_json_or_null(metadata.get('requires_transform'))}
requires_synthesis={_json_or_null(metadata.get('requires_synthesis'))}
requires_all_tool_facts={_json_or_null(metadata.get('requires_all_tool_facts'))}
must_acknowledge_missing_data={_json_or_null(metadata.get('must_acknowledge_missing_data'))}"""

def serialize_state_from_object(input_obj: Dict[str, Any]) -> str:
    if SERIALIZER_VERSION == "serialize_state_v2":
        return serialize_state_v2(input_obj)
    return serialize_state_v1(input_obj)

def serialize_state(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
    scoring_metadata: Optional[Dict[str, Any]] = None,
) -> str:
    return serialize_state_from_object(build_input_object(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
        recent_errors=recent_errors,
        scoring_metadata=scoring_metadata,
    ))

def serialize_final_response_state_v1(input_obj: Dict[str, Any]) -> str:
    ws = input_obj["workflow_state"]
    metadata = input_obj.get("metadata") or {}
    tool_results = "\n".join(
        f"{r.get('tool_name', '')}: {json.dumps(str(r.get('content', '')), ensure_ascii=False)}"
        for r in input_obj.get("tool_results", [])
    )
    return f"""SCHEMA_VERSION:
{input_obj['schema_version']}

USER_REQUEST:
{input_obj['user_request']}

WORKFLOW_STATE:
required_steps={ws.get('required_steps', [])}
completed_steps={ws.get('completed_steps', [])}
pending_steps={ws.get('pending_steps', [])}
terminal_tools={ws.get('terminal_tools', [])}
recent_errors={ws.get('recent_errors', [])}

REQUIRED_FACTS:
{input_obj.get('required_facts', [])}

TOOL_TRACE:
{input_obj.get('tool_trace', [])}

TOOL_RESULTS:
{tool_results}

CANDIDATE_FINAL_RESPONSE:
{input_obj.get('candidate_final_response', '')}

SCORING_METADATA:
scenario_family={_json_or_null(metadata.get('scenario_family'))}
requires_transform={_json_or_null(metadata.get('requires_transform'))}
requires_synthesis={_json_or_null(metadata.get('requires_synthesis'))}
requires_all_tool_facts={_json_or_null(metadata.get('requires_all_tool_facts'))}
must_acknowledge_missing_data={_json_or_null(metadata.get('must_acknowledge_missing_data'))}""".strip()

def make_row(
    source: str,
    label: str,
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    rank_score: float,
    metadata: Optional[Dict[str, Any]] = None,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
    group_id: Optional[str] = None,
    scoring_metadata: Optional[Dict[str, Any]] = None,
) -> VerifierRow:
    metadata = dict(metadata or {})
    scoring_metadata = scoring_metadata or metadata.get("scoring_metadata") or infer_scoring_metadata(metadata.get("scenario_family"))
    group_id = group_id or metadata.get("example_group_id") or stable_id(source, user_request, tools)
    metadata["example_group_id"] = group_id
    metadata["toolset_hash"] = stable_toolset_hash(tools)
    metadata["input_schema_version"] = INPUT_SCHEMA_VERSION
    metadata["serializer_version"] = SERIALIZER_VERSION
    metadata["scoring_metadata"] = scoring_metadata

    input_obj = build_input_object(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
        recent_errors=recent_errors,
        scoring_metadata=scoring_metadata,
    )
    input_text = serialize_state_from_object(input_obj)
    rid = stable_id(source, label, user_request, candidate, group_id, required_steps, completed_steps, pending_steps, terminal_tools)
    return VerifierRow(rid, source, input_text, label, float(rank_score), metadata)

def make_final_response_row(
    source: str,
    label: str,
    user_request: str,
    candidate_final_response: str,
    rank_score: float,
    workflow_state: Optional[Dict[str, Any]] = None,
    required_facts: Optional[List[str]] = None,
    tool_trace: Optional[List[str]] = None,
    tool_results: Optional[List[Dict[str, str]]] = None,
    metadata: Optional[Dict[str, Any]] = None,
    group_id: Optional[str] = None,
) -> FinalResponseVerifierRow:
    metadata = dict(metadata or {})
    scoring_metadata = metadata.get("scoring_metadata") or infer_scoring_metadata(metadata.get("scenario_family"), requires_synthesis=True)
    group_id = group_id or metadata.get("example_group_id") or stable_id(source, user_request, candidate_final_response)
    metadata["example_group_id"] = group_id
    metadata["input_schema_version"] = FINAL_RESPONSE_INPUT_SCHEMA_VERSION
    metadata["serializer_version"] = FINAL_RESPONSE_SERIALIZER_VERSION
    metadata["scoring_metadata"] = scoring_metadata
    input_obj = {
        "schema_version": FINAL_RESPONSE_INPUT_SCHEMA_VERSION,
        "user_request": user_request,
        "workflow_state": workflow_state or {"required_steps": [], "completed_steps": [], "pending_steps": [], "terminal_tools": ["respond"], "recent_errors": []},
        "required_facts": required_facts or [],
        "tool_trace": tool_trace or [],
        "tool_results": tool_results or [],
        "candidate_final_response": candidate_final_response,
        "metadata": scoring_metadata,
    }
    input_text = serialize_final_response_state_v1(input_obj)
    rid = stable_id(source, label, user_request, candidate_final_response, group_id, required_facts, tool_trace)
    return FinalResponseVerifierRow(rid, source, input_text, label, float(rank_score), metadata)

# Save fixed serializer fixtures for Rust and artifact consumers.
SERIALIZER_FIXTURE = build_input_object(
    user_request="Generate a sales report from the Q4 2024 dataset.",
    tools=[
        {"name": "fetch_sales_data", "description": "Fetch sales data.", "parameters": {"type": "object", "properties": {"quarter": {"type": "integer"}, "year": {"type": "integer"}}, "required": ["quarter", "year"]}},
        {"name": "analyze_sales", "description": "Analyze sales.", "parameters": {"type": "object", "properties": {}}},
        {"name": "report", "description": "Produce final report.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
    ],
    candidate={"name": "report", "arguments": {"summary": "Done."}},
    required_steps=["fetch_sales_data", "analyze_sales"],
    completed_steps=[],
    pending_steps=["fetch_sales_data", "analyze_sales"],
    terminal_tools=["report"],
    schema_version=TOOLCALL_INPUT_SCHEMA_VERSION_V1,
)
(DATA_DIR / "serializer_fixture.json").write_text(json.dumps({
    "input": SERIALIZER_FIXTURE,
    "serialized": serialize_state_v1(SERIALIZER_FIXTURE),
}, indent=2))

SERIALIZER_FIXTURE_V2 = build_input_object(
    user_request=SERIALIZER_FIXTURE["user_request"],
    tools=SERIALIZER_FIXTURE["available_tools"],
    candidate=SERIALIZER_FIXTURE["candidate_call"],
    required_steps=SERIALIZER_FIXTURE["workflow_state"]["required_steps"],
    completed_steps=SERIALIZER_FIXTURE["workflow_state"]["completed_steps"],
    pending_steps=SERIALIZER_FIXTURE["workflow_state"]["pending_steps"],
    terminal_tools=SERIALIZER_FIXTURE["workflow_state"]["terminal_tools"],
    scoring_metadata=infer_scoring_metadata("argument_transformation"),
    schema_version=TOOLCALL_INPUT_SCHEMA_VERSION_V2,
)
(DATA_DIR / "serializer_fixture_v2.json").write_text(json.dumps({
    "input": SERIALIZER_FIXTURE_V2,
    "serialized": serialize_state_v2(SERIALIZER_FIXTURE_V2),
}, indent=2))
print(serialize_state_from_object(SERIALIZER_FIXTURE_V2 if USE_SERIALIZER_V2 else SERIALIZER_FIXTURE))


## 3. Dataset loaders

These loaders normalize several public function-calling datasets into:

```python
{
  "source": str,
  "user_request": str,
  "tools": list[dict],
  "gold_calls": list[dict],
  "group_id": str
}
```

In [ ]:
def try_json_loads(x: Any) -> Any:
    if isinstance(x, (dict, list)):
        return x
    if x is None:
        return None
    if not isinstance(x, str):
        return x
    s = x.strip()
    if not s:
        return None
    try:
        return json.loads(s)
    except Exception:
        pass
    try:
        return ast.literal_eval(s)
    except Exception:
        return x

def get_first_text(record: Dict[str, Any], keys: List[str]) -> str:
    for k in keys:
        v = record.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def balanced_json_objects(text: str, start_pos: int = 0) -> List[str]:
    objects = []
    i = start_pos
    while i < len(text):
        if text[i] != "{":
            i += 1
            continue
        start = i
        depth = 0
        in_str = False
        escape = False
        while i < len(text):
            ch = text[i]
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = not in_str
            elif not in_str:
                if ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        objects.append(text[start:i + 1])
                        break
            i += 1
        i += 1
    return objects

def extract_json_after_marker(text: str, marker: str) -> List[Any]:
    out = []
    if not isinstance(text, str) or marker not in text:
        return out
    cursor = 0
    while True:
        idx = text.find(marker, cursor)
        if idx < 0:
            break
        start = idx + len(marker)
        objs = balanced_json_objects(text, start)
        if objs:
            parsed = try_json_loads(objs[0])
            out.append(parsed)
            cursor = text.find(objs[0], start) + len(objs[0])
        else:
            cursor = start
    return out

def extract_first_user_from_chat(chat: Any) -> str:
    chat = try_json_loads(chat)
    if isinstance(chat, list):
        for m in chat:
            if isinstance(m, dict) and m.get("from") in ("user", "human"):
                return str(m.get("value", "")).strip()
    if not isinstance(chat, str):
        return ""
    m = re.search(r"USER:\s*(.*?)(?:\s+ASSISTANT:|<\|endoftext\|>|$)", chat, flags=re.S)
    if m:
        return m.group(1).strip()
    return ""

def extract_messages_text(messages: Any) -> str:
    messages = try_json_loads(messages)
    if isinstance(messages, list):
        parts = []
        for m in messages:
            if not isinstance(m, dict):
                continue
            role = m.get("role", m.get("from", ""))
            content = m.get("content", m.get("value", ""))
            if isinstance(content, list):
                content = " ".join(str(c) for c in content)
            if role in ("user", "human") and content:
                parts.append(str(content))
        return "\n".join(parts).strip()
    return extract_first_user_from_chat(messages)

def normalize_tool_object(t: Any) -> Optional[Dict[str, Any]]:
    t = try_json_loads(t)
    if not isinstance(t, dict):
        return None
    if t.get("type") == "function" and isinstance(t.get("function"), dict):
        f = t["function"]
        return {
            "name": f.get("name", "unknown_tool"),
            "description": f.get("description", ""),
            "parameters": f.get("parameters", {"type": "object", "properties": {}}),
        }
    name = t.get("name") or t.get("tool_name") or t.get("api_name") or t.get("function_name")
    if not name and isinstance(t.get("function"), dict):
        name = t["function"].get("name")
    if not name:
        return None
    params = t.get("parameters") or t.get("parameter") or t.get("args") or t.get("arguments_schema") or t.get("input_schema") or {"type": "object", "properties": {}}
    params = try_json_loads(params)
    if not isinstance(params, dict):
        params = {"type": "object", "properties": {}}
    return {"name": str(name), "description": str(t.get("description") or t.get("desc") or ""), "parameters": params}

def extract_tools_from_text(text: Any) -> List[Dict[str, Any]]:
    if not isinstance(text, str):
        return []
    tools = []
    for obj_text in balanced_json_objects(text):
        obj = try_json_loads(obj_text)
        if isinstance(obj, list):
            for item in obj:
                tool = normalize_tool_object(item)
                if tool:
                    tools.append(tool)
        else:
            tool = normalize_tool_object(obj)
            if tool:
                tools.append(tool)
    seen = set()
    deduped = []
    for t in tools:
        name = t.get("name")
        if name and name not in seen:
            seen.add(name)
            deduped.append(t)
    return deduped

def extract_tools(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidate_keys = ["tools", "functions", "function", "tool", "apis", "api_list", "function_list", "available_tools", "function_definition"]
    for k in candidate_keys:
        if k not in record:
            continue
        raw = try_json_loads(record.get(k))
        if isinstance(raw, dict):
            if any(key in raw for key in ["name", "function", "tool_name", "api_name"]):
                maybe = normalize_tool_object(raw)
                return [maybe] if maybe else []
            for nested_key in ["tools", "functions", "apis"]:
                if isinstance(raw.get(nested_key), list):
                    raw = raw[nested_key]
                    break
        if isinstance(raw, list):
            out = []
            for item in raw:
                t = normalize_tool_object(item)
                if t:
                    out.append(t)
            if out:
                return out
    for text_key in ["system", "chat", "conversation", "conversations"]:
        out = extract_tools_from_text(record.get(text_key))
        if out:
            return out
    return []

def normalize_call_object(c: Any) -> Optional[Dict[str, Any]]:
    c = try_json_loads(c)
    if isinstance(c, list) and c:
        return normalize_call_object(c[0])
    if not isinstance(c, dict):
        return None
    if isinstance(c.get("function"), dict):
        f = c["function"]
        args = try_json_loads(f.get("arguments", {}))
        if not isinstance(args, dict):
            args = {"_raw": args}
        return {"name": f.get("name", "unknown_tool"), "arguments": args}
    name = c.get("name") or c.get("tool_name") or c.get("function_name") or c.get("api_name")
    args = c.get("arguments", c.get("args", c.get("parameters", c.get("input", {}))))
    args = try_json_loads(args)
    if not isinstance(args, dict):
        args = {"_raw": args}
    if name:
        return {"name": str(name), "arguments": args}
    if isinstance(c.get("tool"), dict):
        return normalize_call_object(c["tool"])
    return None

def parse_toolace_call_text(value: str) -> Optional[Dict[str, Any]]:
    if not isinstance(value, str):
        return None
    m = re.search(r"\[([A-Za-z0-9_ ./-]+)\((.*?)\)\]", value)
    if not m:
        return None
    name = m.group(1).strip()
    raw_args = m.group(2).strip()
    args = {}
    if raw_args:
        for part in re.split(r",\s*(?=[A-Za-z_][A-Za-z0-9_]*\s*=)", raw_args):
            if "=" not in part:
                continue
            k, v = part.split("=", 1)
            k = k.strip()
            v = v.strip()
            try:
                args[k] = ast.literal_eval(v)
            except Exception:
                args[k] = v.strip('"').strip("'")
    return {"name": name, "arguments": args}

def extract_gold_calls(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidate_keys = ["tool_calls", "function_call", "function_calls", "answers", "answer", "assistant", "calls", "api_calls", "gold_calls"]
    calls = []
    for k in candidate_keys:
        if k not in record:
            continue
        raw = try_json_loads(record.get(k))
        if isinstance(raw, list):
            for item in raw:
                call = normalize_call_object(item)
                if call:
                    calls.append(call)
        else:
            call = normalize_call_object(raw)
            if call:
                calls.append(call)
        if calls:
            return calls
    for text_key in ["chat", "conversation", "conversations", "messages"]:
        text = record.get(text_key)
        if isinstance(text, str):
            for marker in ["<functioncall>", "<tool_call>"]:
                for obj in extract_json_after_marker(text, marker):
                    call = normalize_call_object(obj)
                    if call:
                        calls.append(call)
        conv = try_json_loads(text)
        if isinstance(conv, list):
            for m in conv:
                if not isinstance(m, dict):
                    continue
                role = m.get("role", m.get("from", ""))
                content = m.get("content", m.get("value", ""))
                if role in ("assistant", "gpt"):
                    call = normalize_call_object(m.get("tool_calls") or m.get("function_call"))
                    if call:
                        calls.append(call)
                    parsed = parse_toolace_call_text(str(content))
                    if parsed:
                        calls.append(parsed)
    return calls

def normalize_record(record: Dict[str, Any], source: str, index: int) -> Optional[Dict[str, Any]]:
    user_request = (
        get_first_text(record, ["query", "question", "instruction", "user", "prompt", "input"])
        or extract_messages_text(record.get("messages"))
        or extract_messages_text(record.get("conversation"))
        or extract_messages_text(record.get("conversations"))
        or extract_first_user_from_chat(record.get("chat"))
    )
    tools = extract_tools(record)
    gold_calls = extract_gold_calls(record)
    if not tools and gold_calls:
        tools = [
            {"name": c["name"], "description": "Tool parsed from conversation.", "parameters": {"type": "object", "properties": {}}}
            for c in gold_calls if c.get("name")
        ]
    if not user_request or not tools or not gold_calls:
        return None
    return {
        "source": source,
        "user_request": user_request,
        "tools": tools,
        "gold_calls": gold_calls,
        "group_id": stable_id(source, index, user_request, tools, gold_calls),
    }

def load_source_dataset(name: str, split: str = "train", max_rows: int = MAX_PER_SOURCE, **kwargs) -> List[Dict[str, Any]]:
    print(f"Loading {name}...")
    try:
        ds = load_dataset(name, split=split, streaming=False, **kwargs)
    except Exception as e:
        print(f"  load_dataset failed for {name}: {type(e).__name__}: {e}")
        return []
    if max_rows:
        ds = ds.select(range(min(max_rows, len(ds))))
    rows = []
    for i, rec in enumerate(tqdm(ds, desc=f"normalizing {name}")):
        norm = normalize_record(dict(rec), source=name, index=i)
        if norm:
            rows.append(norm)
    print(f"  normalized {len(rows)} usable rows out of {len(ds)}")
    if len(rows) == 0 and len(ds) > 0:
        sample = dict(ds[0])
        print("  No rows parsed. Sample keys:", list(sample.keys()))
        print("  Sample preview:", json.dumps({k: str(v)[:500] for k, v in sample.items()}, indent=2))
    return rows

In [ ]:
#@title Load public function-calling sources
# Some datasets change schema or require auth. Failed datasets are skipped.
PUBLIC_SOURCES = [
    ("glaiveai/glaive-function-calling-v2", {"split": "train"}),
    ("Team-ACE/ToolACE", {"split": "train"}),
    ("Salesforce/xlam-function-calling-60k", {"split": "train"}),
    # BFCL is better used as a separate eval harness from its JSONL files.
]

normalized_examples = []
source_counts = {}
for dataset_name, kwargs in PUBLIC_SOURCES:
    rows = load_source_dataset(dataset_name, max_rows=MAX_PER_SOURCE, **kwargs)
    normalized_examples.extend(rows)
    source_counts[dataset_name] = len(rows)

print("Usable rows by source:")
for k, v in source_counts.items():
    print(f"  {k}: {v}")
print("Total normalized examples:", len(normalized_examples))

if normalized_examples:
    print(json.dumps(normalized_examples[0], indent=2)[:4000])
else:
    print("No public rows parsed. Training will be a smoke test unless Forge traces are added.")

## 4. Forge-specific synthetic and trace rows

Public datasets do not know your workflow contract concepts: required steps, terminal tools, prerequisites, unsafe terminal-plus-work batches, or respond/final tools. Keep these rows modest unless a separate ablation proves they help.

In [ ]:
FORGE_WORKFLOWS = [
    {
        "source": "forge_synthetic",
        "user_request": "Generate a sales report from the Q4 2024 dataset.",
        "tools": [
            {"name": "fetch_sales_data", "description": "Fetch sales data for a given quarter and year.", "parameters": {"type": "object", "properties": {"quarter": {"type": "integer"}, "year": {"type": "integer"}}, "required": ["quarter", "year"]}},
            {"name": "analyze_sales", "description": "Analyze the loaded sales data and produce findings.", "parameters": {"type": "object", "properties": {}}},
            {"name": "report", "description": "Produce the final report from findings.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
        ],
        "required_steps": ["fetch_sales_data", "analyze_sales"],
        "terminal_tools": ["report"],
        "valid_sequence": [
            {"name": "fetch_sales_data", "arguments": {"quarter": 4, "year": 2024}},
            {"name": "analyze_sales", "arguments": {}},
            {"name": "report", "arguments": {"summary": "Revenue grew 23% YoY. Top product: Widget Pro. Weakest region: APAC."}},
        ],
    },
    {
        "source": "forge_synthetic",
        "user_request": "Fetch 10 records and summarize them.",
        "tools": [
            {"name": "fetch", "description": "Fetch a count of records. Count must be a four digit string.", "parameters": {"type": "object", "properties": {"count": {"type": "string"}}, "required": ["count"]}},
            {"name": "summarize", "description": "Summarize fetched records.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
        ],
        "required_steps": ["fetch"],
        "terminal_tools": ["summarize"],
        "valid_sequence": [
            {"name": "fetch", "arguments": {"count": "0010"}},
            {"name": "summarize", "arguments": {"summary": "Fetched 10 records."}},
        ],
    },
]

ARG_TRANSFORMATION_TOOLS = [
    {"name": "list_transactions", "description": "List all expense transactions for a given fiscal quarter and year.", "parameters": {"type": "object", "properties": {"quarter": {"type": "string"}, "year": {"type": "integer"}}, "required": ["quarter", "year"]}},
    {"name": "get_approved_vendors", "description": "Return the canonical list of approved vendor names.", "parameters": {"type": "object", "properties": {}}},
    {"name": "get_vendor_details", "description": "Look up vendor details, status, legal entity, and aliases.", "parameters": {"type": "object", "properties": {"vendor_name": {"type": "string"}}, "required": ["vendor_name"]}},
    {"name": "currency_convert", "description": "Convert an amount between currencies.", "parameters": {"type": "object", "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"}, "to_currency": {"type": "string"}}, "required": ["amount", "from_currency", "to_currency"]}},
    {"name": "submit_audit_report", "description": "Submit transaction IDs, total USD amount, and largest vendor.", "parameters": {"type": "object", "properties": {"transaction_ids": {"type": "string"}, "total_flagged_usd": {"type": "string"}, "top_vendor": {"type": "string"}}, "required": ["transaction_ids", "total_flagged_usd", "top_vendor"]}},
]
ARG_TRANSFORMATION_USER = "Run our Q4 2024 expense audit. Flag any transaction of $5,000 or more from vendors NOT on our approved list. Submit the audit report with: comma-separated transaction IDs, total flagged amount in USD, and the vendor of the single largest flagged transaction."
ARG_TRANSFORMATION_CANONICAL_CALL = {"name": "submit_audit_report", "arguments": {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008", "total_flagged_usd": "$28,980.00", "top_vendor": "Wonka Industries"}}
ARG_TRANSFORMATION_BAD_CALLS = [
    ("skip_currency_conversion", {"transaction_ids": "TX-1001, TX-1005, TX-1008", "total_flagged_usd": "$23,700", "top_vendor": "Wonka Industries"}),
    ("strict_gt_threshold", {"transaction_ids": "TX-1001, TX-1006, TX-1008", "total_flagged_usd": "$23,980", "top_vendor": "Wonka Industries"}),
    ("vendor_alias_overflag", {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008, TX-1009", "total_flagged_usd": "$35,480", "top_vendor": "Wonka Industries"}),
    ("wrong_total", {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008", "total_flagged_usd": "$76,620.00", "top_vendor": "Wonka Industries"}),
    ("wrong_top_vendor", {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008", "total_flagged_usd": "$28,980.00", "top_vendor": "Cyberdyne LLC"}),
]

def build_forge_synthetic_rows() -> List[VerifierRow]:
    rows = []
    for wf_idx, wf in enumerate(FORGE_WORKFLOWS):
        tools = wf["tools"]
        required = wf["required_steps"]
        terminal = wf["terminal_tools"]
        group_id = stable_id("forge_synthetic", wf_idx, wf["user_request"])
        completed = []
        for step_idx, call in enumerate(wf["valid_sequence"]):
            pending = [s for s in required if s not in completed]
            rows.append(make_row("forge_synthetic", "valid", wf["user_request"], tools, call, 1.0, {"generator": "forge_synthetic", "step_idx": step_idx}, required, completed.copy(), pending, terminal, group_id=group_id))
            if call["name"] in required:
                completed.append(call["name"])
        rows.append(make_row("forge_synthetic", "premature_terminal", wf["user_request"], tools, wf["valid_sequence"][-1], 0.05, {"generator": "forge_synthetic", "negative_type": "premature_terminal"}, required, [], required, terminal, group_id=group_id))
        if len(wf["valid_sequence"]) > 1:
            rows.append(make_row("forge_synthetic", "unsafe_parallel_batch", wf["user_request"], tools, [wf["valid_sequence"][0], wf["valid_sequence"][-1]], 0.02, {"generator": "forge_synthetic", "negative_type": "unsafe_parallel_batch"}, required, [], required, terminal, group_id=group_id))
    return rows

def build_argument_semantic_rows() -> List[VerifierRow]:
    rows = []
    metadata = {"generator": "forge_argument_semantic", "scenario_family": "argument_transformation"}
    group_id = stable_id("forge_argument_semantic", ARG_TRANSFORMATION_USER)
    rows.append(make_row("forge_argument_semantic", "valid", ARG_TRANSFORMATION_USER, ARG_TRANSFORMATION_TOOLS, ARG_TRANSFORMATION_CANONICAL_CALL, 1.0, metadata, ["list_transactions", "get_approved_vendors"], ["list_transactions", "get_approved_vendors"], [], ["submit_audit_report"], group_id=group_id))
    for negative_type, args in ARG_TRANSFORMATION_BAD_CALLS:
        rows.append(make_row("forge_argument_semantic", "wrong_arguments_semantic", ARG_TRANSFORMATION_USER, ARG_TRANSFORMATION_TOOLS, {"name": "submit_audit_report", "arguments": args}, 0.05, dict(metadata, negative_type=negative_type), ["list_transactions", "get_approved_vendors"], ["list_transactions", "get_approved_vendors"], [], ["submit_audit_report"], group_id=group_id))
    return rows

def build_guardrail_augmented_rows(n_per_label: int = FORGE_AUGMENTATION_PER_LABEL) -> List[VerifierRow]:
    rows = []
    for i in range(n_per_label):
        base = random.choice(FORGE_WORKFLOWS)
        tools = base["tools"]
        required = base["required_steps"]
        terminal = base["terminal_tools"]
        clear_user = base["user_request"]
        missing_user = "Do the thing with the dataset, but I do not know which quarter, year, or count."
        fetch = required[0]
        fetch_tool = next(t for t in tools if t["name"] == fetch)
        req_keys = fetch_tool.get("parameters", {}).get("required", []) or []
        required_args = {k: (4 if k == "quarter" else 2024 if k == "year" else "0010") for k in req_keys}
        terminal_tool = terminal[0]
        group_id = stable_id("forge_augmented", i, clear_user)
        rows.append(make_row("forge_augmented", "premature_terminal", clear_user, tools, {"name": terminal_tool, "arguments": {"summary": "Done."}}, 0.05, {"generator": "forge_augmented", "negative_type": "premature_terminal"}, required, [], required, terminal, group_id=group_id))
        rows.append(make_row("forge_augmented", "unsafe_parallel_batch", clear_user, tools, [{"name": fetch, "arguments": required_args}, {"name": terminal_tool, "arguments": {"summary": "Done."}}], 0.02, {"generator": "forge_augmented", "negative_type": "unsafe_parallel_batch"}, required, [], required, terminal, group_id=group_id))
        guessed_args = dict(required_args)
        if guessed_args:
            guessed_args[list(guessed_args.keys())[0]] = "UNKNOWN"
        rows.append(make_row("forge_augmented", "needs_clarification", missing_user, tools, {"name": fetch, "arguments": guessed_args}, 0.25, {"generator": "forge_augmented", "negative_type": "needs_clarification"}, required, [], required, terminal, group_id=group_id))
    return rows

forge_rows = build_forge_synthetic_rows() + build_argument_semantic_rows()
if ENABLE_FORGE_AUGMENTATION:
    forge_rows += build_guardrail_augmented_rows()

print("Forge rows:", len(forge_rows))
print(pd.Series([r.label for r in forge_rows]).value_counts())


In [ ]:
#@title Optional: load Forge eval/proxy trace JSONL and hard-negative files
# In Colab, upload python_oracle_colab_trace.jsonl and optional *.hard_negatives.jsonl siblings to /content.
FORGE_TRACE_JSONL = "/content/python_oracle_colab_trace.jsonl"  #@param {type:"string"}
FORGE_HARD_NEGATIVE_GLOB = "/content/*.hard_negatives.jsonl"  #@param {type:"string"}

GROUNDED_SYNTHESIS_USER = "We have an open senior backend engineering role and four candidates in the pool. Review the role, evaluate the candidates, and submit your hiring decision with a brief rationale explaining why you selected (and did not select) the candidates you considered."
GROUNDED_SYNTHESIS_FACTS = ["Aisha Nakamura", "Sarah blocked by non-compete", "James blocked by visa or sponsorship", "Diana role-track mismatch"]
GROUNDED_SYNTHESIS_POSITIVE = "Selected Aisha Nakamura. Sarah Chen is blocked by the Stripe non-compete timeline, James Patel is blocked by visa sponsorship constraints, and Diana Kim is misaligned with the IC role track. Aisha has relevant payments experience and can start immediately."

def tools_from_trace(tool_names: List[str]) -> List[Dict[str, Any]]:
    seen = []
    for name in tool_names:
        if name and name not in seen:
            seen.append(name)
    return [{"name": str(n), "description": "Tool observed in Forge eval trace.", "parameters": {"type": "object", "properties": {}}} for n in seen]

def trace_terminal_candidate(rec: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    tool_names = rec.get("tool_sequence") or []
    tool_args = rec.get("tool_args") or []
    if not tool_names:
        return None
    name = tool_names[-1]
    args = tool_args[-1] if tool_args and len(tool_args) >= len(tool_names) else {}
    return {"name": name, "arguments": args if isinstance(args, dict) else {"_raw": args}}

def load_forge_trace_rows(path: str) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    if not path or not Path(path).exists():
        return [], []
    tool_rows: List[VerifierRow] = []
    final_rows: List[FinalResponseVerifierRow] = []
    with open(path) as f:
        for idx, line in enumerate(f):
            if not line.strip():
                continue
            rec = json.loads(line)
            scenario = str(rec.get("scenario") or "")
            user_request = rec.get("user_message") or rec.get("prompt") or rec.get("request") or ""
            tool_names = rec.get("tool_sequence") or []
            tool_args = rec.get("tool_args") or []
            tools = tools_from_trace(tool_names)
            if not user_request or not tools:
                continue
            group_id = stable_id("forge_trace", scenario, rec.get("run"), user_request)
            failed_accuracy = rec.get("accuracy") is False or rec.get("success") is False
            terminal = trace_terminal_candidate(rec)
            family_meta = infer_scoring_metadata(scenario)
            if failed_accuracy and terminal and "argument_transformation" in scenario:
                tool_rows.append(make_row(
                    "forge_trace", "wrong_arguments_semantic", user_request, tools, terminal, 0.05,
                    {"generator": "forge_trace", "trace_index": idx, "scenario_family": scenario, "failure_kind": "accuracy_false"},
                    required_steps=["list_transactions", "get_approved_vendors"], completed_steps=["list_transactions", "get_approved_vendors"], pending_steps=[], terminal_tools=[terminal["name"]], group_id=group_id, scoring_metadata=family_meta,
                ))
                tool_rows.append(make_row(
                    "forge_trace", "valid", ARG_TRANSFORMATION_USER, ARG_TRANSFORMATION_TOOLS, ARG_TRANSFORMATION_CANONICAL_CALL, 1.0,
                    {"generator": "forge_trace_canonical_positive", "scenario_family": scenario},
                    required_steps=["list_transactions", "get_approved_vendors"], completed_steps=["list_transactions", "get_approved_vendors"], pending_steps=[], terminal_tools=["submit_audit_report"], group_id=group_id, scoring_metadata=family_meta,
                ))
            else:
                for call_idx, name in enumerate(tool_names):
                    args = tool_args[call_idx] if call_idx < len(tool_args) else {}
                    label = "valid"
                    if bool(rec.get("required_step_mismatch")) or rec.get("missing_required_steps"):
                        label = "missing_prerequisite"
                    error_kind = str(rec.get("error_kind") or rec.get("error_type") or "")
                    if "unknown" in error_kind.lower():
                        label = "unknown_tool"
                    tool_rows.append(make_row(
                        "forge_trace", label, user_request, tools, {"name": name, "arguments": args if isinstance(args, dict) else {"_raw": args}},
                        1.0 if label == "valid" else 0.2,
                        {"generator": "forge_trace", "trace_index": idx, "error_kind": error_kind, "scenario_family": scenario},
                        group_id=group_id, scoring_metadata=family_meta,
                    ))
            if failed_accuracy and "grounded_synthesis" in scenario:
                candidate_final = str(rec.get("final_text") or "")
                final_rows.append(make_final_response_row(
                    "forge_trace", "contradicts_tool_result", user_request, candidate_final, 0.05,
                    workflow_state={"required_steps": ["get_open_role", "get_candidate_pool"], "completed_steps": ["get_open_role", "get_candidate_pool"], "pending_steps": [], "terminal_tools": ["submit_hiring_decision"], "recent_errors": []},
                    required_facts=GROUNDED_SYNTHESIS_FACTS,
                    tool_trace=tool_names,
                    tool_results=[{"tool_name": n, "content": compact_json(a, 500)} for n, a in zip(tool_names, tool_args)],
                    metadata={"generator": "forge_trace", "scenario_family": scenario, "failure_kind": "accuracy_false"},
                    group_id=group_id,
                ))
                final_rows.append(make_final_response_row(
                    "forge_trace", "valid_final_response", user_request, GROUNDED_SYNTHESIS_POSITIVE, 1.0,
                    workflow_state={"required_steps": ["get_open_role", "get_candidate_pool"], "completed_steps": ["get_open_role", "get_candidate_pool"], "pending_steps": [], "terminal_tools": ["submit_hiring_decision"], "recent_errors": []},
                    required_facts=GROUNDED_SYNTHESIS_FACTS,
                    tool_trace=tool_names,
                    tool_results=[{"tool_name": n, "content": compact_json(a, 500)} for n, a in zip(tool_names, tool_args)],
                    metadata={"generator": "forge_trace_canonical_positive", "scenario_family": scenario},
                    group_id=group_id,
                ))
    return tool_rows, final_rows

def load_hard_negative_rows(pattern: str) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    tool_rows: List[VerifierRow] = []
    final_rows: List[FinalResponseVerifierRow] = []
    for file_path in glob.glob(pattern or ""):
        with open(file_path) as f:
            for idx, line in enumerate(f):
                if not line.strip():
                    continue
                rec = json.loads(line)
                outcome = rec.get("outcome") or {}
                scenario = str(outcome.get("scenario_family") or outcome.get("scenario") or "")
                group_id = stable_id("hard_negative", file_path, idx, scenario, outcome.get("run"))
                if rec.get("kind") == "tool_call":
                    candidate = rec.get("candidate") or {}
                    seq = candidate.get("tool_sequence") or []
                    args = candidate.get("tool_args") or []
                    terminal = {"name": seq[-1], "arguments": args[-1] if args else {}} if seq else {"name": "unknown_tool", "arguments": {}}
                    label = "wrong_arguments_semantic" if "argument_transformation" in scenario else "wrong_tool_semantic"
                    tool_rows.append(make_row(
                        "forge_hard_negative", label, outcome.get("user_request") or "Forge eval hard negative", tools_from_trace(seq), terminal, 0.05,
                        {"generator": "forge_hard_negative", "scenario_family": scenario, "source_file": file_path},
                        group_id=group_id, scoring_metadata=infer_scoring_metadata(scenario),
                    ))
                elif rec.get("kind") == "final_response":
                    candidate = rec.get("candidate") or {}
                    final_rows.append(make_final_response_row(
                        "forge_hard_negative", "contradicts_tool_result", outcome.get("user_request") or "Forge eval final response", str(candidate.get("final_text") or ""), 0.05,
                        required_facts=GROUNDED_SYNTHESIS_FACTS if "grounded_synthesis" in scenario else [],
                        metadata={"generator": "forge_hard_negative", "scenario_family": scenario, "source_file": file_path},
                        group_id=group_id,
                    ))
    return tool_rows, final_rows

trace_rows, final_response_trace_rows = load_forge_trace_rows(FORGE_TRACE_JSONL)
hard_negative_rows, final_response_hard_negative_rows = load_hard_negative_rows(FORGE_HARD_NEGATIVE_GLOB)
trace_rows += hard_negative_rows
final_response_rows = final_response_trace_rows + final_response_hard_negative_rows
print("Forge trace/tool hard-negative rows:", len(trace_rows))
print("Final-response rows:", len(final_response_rows))


## 5. Convert public generation rows into verifier/ranker rows

For each gold call, generate a valid candidate plus hard negatives. All candidates derived from the same original example share `example_group_id` so split leakage is reduced.

In [ ]:
def required_args_for_tool(tool: Dict[str, Any]) -> List[str]:
    params = tool.get("parameters") or {}
    req = params.get("required") or []
    return [str(x) for x in req] if isinstance(req, list) else []

def tool_by_name(tools: List[Dict[str, Any]], name: str) -> Optional[Dict[str, Any]]:
    for t in tools:
        if t.get("name") == name:
            return t
    return None

def wrong_tool_candidate(tools: List[Dict[str, Any]], gold_name: str, gold_args: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    others = [t for t in tools if t.get("name") != gold_name]
    if not others:
        return None
    t = random.choice(others)
    return {"name": t.get("name", "unknown_tool"), "arguments": gold_args or {}}

def missing_arg_candidate(tool: Dict[str, Any], call: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    req = required_args_for_tool(tool)
    if not req:
        return None
    args = dict(call.get("arguments") or {})
    removed = False
    for k in req:
        if k in args:
            args.pop(k)
            removed = True
            break
    if not removed:
        args = {}
    return {"name": call["name"], "arguments": args}

def invalid_shape_candidate(call: Dict[str, Any]) -> Dict[str, Any]:
    return {"name": call["name"], "arguments": {"_raw": "not an object or wrong shape"}}

def wrong_argument_candidate(call: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    args = dict(call.get("arguments") or {})
    if not args:
        return None
    key = next(iter(args.keys()))
    value = args[key]
    mutated = dict(args)
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        mutated[key] = value + 1 if value != 0 else 999
    elif isinstance(value, str):
        mutated[key] = value[::-1] if len(value) > 1 else value + "_wrong"
    elif isinstance(value, bool):
        mutated[key] = not value
    else:
        mutated[key] = "wrong_value"
    if mutated == args:
        return None
    return {"name": call["name"], "arguments": mutated}

def make_public_rows(examples: List[Dict[str, Any]]) -> List[VerifierRow]:
    rows = []
    for ex in examples:
        source = ex["source"]
        user_request = ex["user_request"]
        tools = ex["tools"]
        gold_calls = ex["gold_calls"]
        group_id = ex.get("group_id") or stable_id(source, user_request, tools, gold_calls)
        for call in gold_calls:
            if not call.get("name"):
                continue
            call_tool = tool_by_name(tools, call["name"])
            rows.append(make_row(
                source=source,
                label="valid",
                user_request=user_request,
                tools=tools,
                candidate=call,
                rank_score=1.0,
                metadata={"generator": "public_gold", "gold_tool": call["name"]},
                group_id=group_id,
            ))
            wrong = wrong_tool_candidate(tools, call["name"], call.get("arguments") or {})
            if wrong:
                rows.append(make_row(
                    source=source,
                    label="wrong_tool_semantic",
                    user_request=user_request,
                    tools=tools,
                    candidate=wrong,
                    rank_score=0.15,
                    metadata={"generator": "hard_negative", "negative_type": "wrong_tool", "gold_tool": call["name"]},
                    group_id=group_id,
                ))
            if call_tool:
                missing = missing_arg_candidate(call_tool, call)
                if missing:
                    rows.append(make_row(
                        source=source,
                        label="missing_required_args",
                        user_request=user_request,
                        tools=tools,
                        candidate=missing,
                        rank_score=0.10,
                        metadata={"generator": "hard_negative", "negative_type": "missing_required_args", "gold_tool": call["name"]},
                        group_id=group_id,
                    ))
                wrong_args = wrong_argument_candidate(call)
                if wrong_args:
                    rows.append(make_row(
                        source=source,
                        label="wrong_arguments_semantic",
                        user_request=user_request,
                        tools=tools,
                        candidate=wrong_args,
                        rank_score=0.08,
                        metadata={"generator": "hard_negative", "negative_type": "wrong_argument_value", "gold_tool": call["name"]},
                        group_id=group_id,
                    ))
            rows.append(make_row(
                source=source,
                label="invalid_args_schema",
                user_request=user_request,
                tools=tools,
                candidate=invalid_shape_candidate(call),
                rank_score=0.05,
                metadata={"generator": "hard_negative", "negative_type": "invalid_arg_shape", "gold_tool": call["name"]},
                group_id=group_id,
            ))
            rows.append(make_row(
                source=source,
                label="unknown_tool",
                user_request=user_request,
                tools=tools,
                candidate={"name": "unknown_tool", "arguments": call.get("arguments") or {}},
                rank_score=0.01,
                metadata={"generator": "hard_negative", "negative_type": "unknown_tool", "gold_tool": call["name"]},
                group_id=group_id,
            ))
            rows.append(make_row(
                source=source,
                label="malformed_tool_call",
                user_request=user_request,
                tools=tools,
                candidate={"malformed": "missing name/function fields"},
                rank_score=0.01,
                metadata={"generator": "hard_negative", "negative_type": "malformed_tool_call", "gold_tool": call["name"]},
                group_id=group_id,
            ))
            if random.random() < 0.20:
                rows.append(make_row(
                    source=source,
                    label="tool_not_needed",
                    user_request=user_request,
                    tools=tools,
                    candidate={"text_response": "I can answer this directly without tools."},
                    rank_score=0.30,
                    metadata={"generator": "hard_negative", "negative_type": "text_instead_of_tool", "gold_tool": call["name"]},
                    group_id=group_id,
                ))
    return rows

public_rows = make_public_rows(normalized_examples)
all_rows = public_rows + forge_rows + trace_rows
print("public rows:", len(public_rows))
print("forge rows:", len(forge_rows))
print("trace rows:", len(trace_rows))
print("all rows:", len(all_rows))

In [ ]:
#@title Build dataframe, normalize labels, lightly cap large classes

def rows_to_df(rows: List[VerifierRow]) -> pd.DataFrame:
    return pd.DataFrame([asdict(r) for r in rows])

df = rows_to_df(all_rows)
if df.empty:
    raise RuntimeError("No training rows were created. Inspect dataset schemas or add Forge traces.")

df["raw_label"] = df["label"]
df["label"] = df["label"].map(normalize_label)
df["example_group_id"] = df["metadata"].map(lambda m: m.get("example_group_id") if isinstance(m, dict) else None)
df["toolset_hash"] = df["metadata"].map(lambda m: m.get("toolset_hash") if isinstance(m, dict) else None)

print("Raw labels:")
print(df["raw_label"].value_counts(dropna=False))
print("\nTraining labels:")
print(df["label"].value_counts(dropna=False))
print("\nGroups:", df["example_group_id"].nunique())

pieces = []
for label, group in df.groupby("label", sort=False):
    n = min(len(group), MAX_PER_LABEL)
    pieces.append(group.sample(n=n, replace=False, random_state=SEED))

balanced = pd.concat(pieces, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print("\nCapped labels:")
print(balanced["label"].value_counts())
balanced.head()

## 6. Group-safe train/validation/test split

Rows generated from the same source example share a group. The split is by group, not by row, so gold calls and hard negatives from one example do not leak across train and test.

In [ ]:
def group_split(
    frame: pd.DataFrame,
    group_col: str = "example_group_id",
    valid_size: float = VALID_SIZE,
    test_size: float = TEST_SIZE,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    frame = frame.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    if group_col not in frame or frame[group_col].isna().all() or frame[group_col].nunique() < 5:
        print("WARNING: insufficient groups; falling back to stratified row split.")
        strat = frame["label"] if frame["label"].value_counts().min() >= 2 else None
        train_df, tmp_df = train_test_split(frame, test_size=valid_size + test_size, random_state=SEED, stratify=strat)
        tmp_strat = tmp_df["label"] if tmp_df["label"].value_counts().min() >= 2 else None
        valid_df, test_df = train_test_split(tmp_df, test_size=test_size / (valid_size + test_size), random_state=SEED, stratify=tmp_strat)
        return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)

    splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size + test_size, random_state=SEED)
    train_idx, tmp_idx = next(splitter.split(frame, groups=frame[group_col]))
    train_df = frame.iloc[train_idx].copy()
    tmp_df = frame.iloc[tmp_idx].copy()

    tmp_test_fraction = test_size / (valid_size + test_size)
    if tmp_df[group_col].nunique() < 2:
        valid_df = tmp_df.copy()
        test_df = tmp_df.copy()
    else:
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=tmp_test_fraction, random_state=SEED + 1)
        valid_idx, test_idx = next(splitter2.split(tmp_df, groups=tmp_df[group_col]))
        valid_df = tmp_df.iloc[valid_idx].copy()
        test_df = tmp_df.iloc[test_idx].copy()

    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, valid_full_df, test_df = group_split(balanced)

if valid_full_df["example_group_id"].nunique() >= 4:
    cal_splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED + 7)
    calibration_idx, valid_idx = next(cal_splitter.split(valid_full_df, groups=valid_full_df["example_group_id"]))
    calibration_df = valid_full_df.iloc[calibration_idx].copy().reset_index(drop=True)
    valid_df = valid_full_df.iloc[valid_idx].copy().reset_index(drop=True)
else:
    calibration_df = valid_full_df.copy().reset_index(drop=True)
    valid_df = valid_full_df.copy().reset_index(drop=True)

for name, split_df in [("train", train_df), ("calibration", calibration_df), ("valid", valid_df), ("test", test_df)]:
    path = DATA_DIR / f"{name}.jsonl"
    with path.open("w") as f:
        for row in split_df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("\n", name, len(split_df), "groups", split_df["example_group_id"].nunique(), path)
    print(split_df["label"].value_counts().to_string())

# Verify no train/test leakage. Calibration and validation are split from the same validation holdout.
train_groups = set(train_df["example_group_id"])
calibration_groups = set(calibration_df["example_group_id"])
valid_groups = set(valid_df["example_group_id"])
test_groups = set(test_df["example_group_id"])
assert train_groups.isdisjoint(calibration_groups)
assert train_groups.isdisjoint(valid_groups)
assert train_groups.isdisjoint(test_groups)
assert test_groups.isdisjoint(calibration_groups)
assert test_groups.isdisjoint(valid_groups)
if calibration_groups and valid_groups:
    assert calibration_groups.isdisjoint(valid_groups)
print("No train/test/calibration group leakage detected.")

full_dataset_path = DATA_DIR / "toolcall_verifier_dataset.jsonl"
with full_dataset_path.open("w") as f:
    for row in balanced.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("Dataset written:", full_dataset_path)


## 7. Tokenization diagnostics

Your token sample had p90 around 955, p95 around 1106, and p99 around 1327. The proven T4 profile uses `MAX_LENGTH=768` as a speed/quality compromise. On the 98GB runtime, start with `high_vram_quality` (`MAX_LENGTH=1280`) before trying the more expensive `high_vram_full_context` (`MAX_LENGTH=1536`).


In [ ]:
ds = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "train.jsonl"),
        "calibration": str(DATA_DIR / "calibration.jsonl"),
        "validation": str(DATA_DIR / "valid.jsonl"),
        "test": str(DATA_DIR / "test.jsonl"),
    },
)

def add_label_id(ex):
    ex["label"] = normalize_label(ex["label"])
    ex["labels"] = label2id[ex["label"]]
    return ex

ds = ds.map(add_label_id)
ds


In [ ]:
#@title Tokenizer load and token-length sample
# DeBERTa v3 uses SentencePiece. The slow tokenizer warning about byte fallback is expected.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

sample_texts = train_df["input_text"].sample(n=min(2000, len(train_df)), random_state=SEED).tolist()
lengths = pd.Series([len(tokenizer.encode(t, add_special_tokens=True)) for t in tqdm(sample_texts, desc="token lengths")])
print("Token length sample:")
print(lengths.describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

p95 = float(lengths.quantile(0.95)) if len(lengths) else 0
if p95 > MAX_LENGTH:
    print(f"WARNING: p95 token length {p95:.0f} exceeds MAX_LENGTH={MAX_LENGTH}. Consider high_vram_quality/high_vram_full_context or more aggressive tool filtering.")
else:
    print("MAX_LENGTH covers p95 in this sample.")


In [ ]:
#@title Tokenize datasets safely for Trainer

def tokenize_batch(batch):
    encoded = tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # dynamic padding happens in DataCollatorWithPadding
    )
    # Keep only numeric labels. Drop the original string `label` column via remove_columns below.
    encoded["labels"] = [int(x) for x in batch["labels"]]
    return encoded

# Remove all non-model columns so DataCollatorWithPadding never tries to tensorize strings/dicts.
tokenized = ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=ds["train"].column_names,
    desc="Tokenizing",
)

# Ensure labels are int64 and torch-compatible.
tokenized = tokenized.cast_column("labels", Value("int64"))
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized)
print("Train feature keys:", tokenized["train"][0].keys())
print({k: type(v) for k, v in tokenized["train"][0].items()})

# Fail fast before Trainer/DataLoader workers hide the root cause.
sample_batch = [tokenized["train"][i] for i in range(min(4, len(tokenized["train"])))]
collated = data_collator(sample_batch)
print("Collated batch keys:", collated.keys())
print("input_ids:", collated["input_ids"].shape, collated["input_ids"].dtype)
print("attention_mask:", collated["attention_mask"].shape, collated["attention_mask"].dtype)
print("labels:", collated["labels"].shape, collated["labels"].dtype)
assert collated["labels"].dtype in (torch.int64, torch.long)


## 8. Train classifier

The completed T4 run is now the baseline:

| Epoch | Accuracy | Macro F1 | Macro F1 all labels |
|---:|---:|---:|---:|
| 1 | 0.819700 | 0.707825 | 0.566260 |
| 2 | 0.830268 | 0.718929 | 0.575143 |
| 3 | 0.832594 | 0.739476 | 0.591581 |
| 4 | 0.839146 | 0.741871 | 0.593497 |

This means the T4 run was still improving at epoch 4. Keep the proven 4-epoch baseline, use early stopping, and select the best checkpoint by `macro_f1`. For the 98GB GPU, use `high_vram_fast` first, then `high_vram_quality`.


In [ ]:
#@title Model, metrics, class weights, and Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label={int(k): v for k, v in id2label.items()},
    label2id=label2id,
)

# Align model config with tokenizer special tokens when needed.
if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
if tokenizer.eos_token_id is not None:
    model.config.eos_token_id = tokenizer.eos_token_id
if tokenizer.bos_token_id is not None:
    model.config.bos_token_id = tokenizer.bos_token_id

ALL_LABEL_IDS = list(range(len(LABELS)))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    present_precision, present_recall, present_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    full_precision, full_recall, full_f1, _ = precision_recall_fscore_support(
        labels, preds, labels=ALL_LABEL_IDS, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "macro_precision": present_precision,
        "macro_recall": present_recall,
        "macro_f1": present_f1,
        "macro_precision_all_labels": full_precision,
        "macro_recall_all_labels": full_recall,
        "macro_f1_all_labels": full_f1,
    }

def build_class_weights(train_frame: pd.DataFrame) -> Optional[torch.Tensor]:
    if not USE_CLASS_WEIGHTS:
        return None
    counts = train_frame["label"].value_counts()
    weights = []
    for label in LABELS:
        count = counts.get(label, 0)
        if count <= 0:
            weights.append(0.0)
        else:
            raw = len(train_frame) / (len(LABELS) * count)
            weights.append(min(float(raw), MAX_CLASS_WEIGHT))
    weights = np.array(weights, dtype=np.float32)
    nonzero = weights[weights > 0]
    if len(nonzero):
        weights[weights > 0] = weights[weights > 0] / nonzero.mean()
    return torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights: Optional[torch.Tensor] = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is None:
            loss = outputs.get("loss")
        else:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

class_weights = build_class_weights(train_df)
if class_weights is None:
    print("Class weights: disabled")
else:
    print("Class weights:")
    for label, weight in zip(LABELS, class_weights.tolist()):
        print(f"  {label:28s} {weight:.3f}")

print("Training profile:", RUN_PROFILE)
print("Training size:", len(tokenized["train"]), "validation size:", len(tokenized["validation"]))
print("Max length:", MAX_LENGTH)
print("Per-device batch:", TRAIN_BATCH_SIZE, "grad accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Optimizer:", OPTIMIZER)
print("Gradient checkpointing:", GRADIENT_CHECKPOINTING)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=USE_FP16,
    bf16=USE_BF16,
    tf32=USE_TF32,
    optim=OPTIMIZER,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
    dataloader_num_workers=DATALOADER_NUM_WORKERS if USE_CUDA else 0,
    dataloader_pin_memory=USE_CUDA,
    group_by_length=GROUP_BY_LENGTH,
    save_total_limit=3,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_result = trainer.train()
trainer.save_state()

# Build a compact run summary for artifact selection, Hub upload, and Rust-side provenance.
eval_history = [dict(row) for row in trainer.state.log_history if "eval_macro_f1" in row]
best_eval = max(eval_history, key=lambda row: row.get("eval_macro_f1", float("-inf")), default={})
training_run_summary = {
    "schema_version": "toolcall-verifier-training-run/v1",
    "base_model": BASE_MODEL,
    "label_mode": LABEL_MODE,
    "requested_gpu_profile": GPU_PROFILE,
    "run_profile": RUN_PROFILE,
    "profile_config": CFG,
    "gpu_info": GPU_INFO,
    "precision_flags": {"fp16": USE_FP16, "bf16": USE_BF16, "tf32": USE_TF32},
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "max_per_source": MAX_PER_SOURCE,
    "train_rows": len(tokenized["train"]),
    "validation_rows": len(tokenized["validation"]),
    "test_rows": len(tokenized["test"]),
    "num_epochs_requested": NUM_EPOCHS,
    "train_result_metrics": train_result.metrics,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric_name": training_args.metric_for_best_model,
    "best_metric": trainer.state.best_metric,
    "best_eval_log": best_eval,
    "eval_history": eval_history,
    "t4_proven_baseline": T4_PROVEN_BASELINE,
    "created_unix": int(time.time()),
}
(DATA_DIR / "training_run_summary.json").write_text(json.dumps(training_run_summary, indent=2))
print("Training run summary:")
print(json.dumps(training_run_summary, indent=2))


In [ ]:
#@title Evaluate calibration, validation, and test splits

split_logits = {}
split_label_ids = {}

def score_split(split_name: str) -> pd.DataFrame:
    pred_out = trainer.predict(tokenized[split_name])
    logits = pred_out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(logits, axis=-1)
    labels_arr = pred_out.label_ids
    split_logits[split_name] = logits
    split_label_ids[split_name] = labels_arr
    raw = ds[split_name].to_pandas()
    raw["true_label"] = [id2label[int(i)] for i in labels_arr]
    raw["pred_label"] = [id2label[int(i)] for i in preds]
    raw["confidence"] = probs.max(axis=-1)
    for idx, label in id2label.items():
        raw[f"prob_{label}"] = probs[:, int(idx)]
        raw[f"logit_{label}"] = logits[:, int(idx)]
    raw["correct"] = raw["true_label"] == raw["pred_label"]
    return raw

test_metrics = trainer.evaluate(tokenized["test"])
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))

calibration_scored = score_split("calibration")
valid_scored = score_split("validation")
test_scored = score_split("test")

def print_report(scored: pd.DataFrame, name: str):
    y_true = [label2id[x] for x in scored["true_label"]]
    y_pred = [label2id[x] for x in scored["pred_label"]]
    print(f"\n{name} classification report")
    print(classification_report(y_true, y_pred, labels=ALL_LABEL_IDS, target_names=LABELS, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=ALL_LABEL_IDS)
    display(pd.DataFrame(cm, index=LABELS, columns=LABELS))
    print("Per-source accuracy:")
    display(scored.groupby("source").agg(rows=("correct", "size"), accuracy=("correct", "mean"), avg_confidence=("confidence", "mean")).sort_values("rows", ascending=False))
    print("Per-label accuracy:")
    display(scored.groupby("true_label").agg(rows=("correct", "size"), accuracy=("correct", "mean"), avg_confidence=("confidence", "mean")).sort_values("rows", ascending=False))

print_report(test_scored, "test")

mistakes = test_scored[test_scored["correct"] == False].sort_values("confidence", ascending=False)[
    ["true_label", "pred_label", "confidence", "source", "input_text", "metadata"]
]
print("High-confidence mistakes:")
display(mistakes.head(25))

valid_false_block_rates = {}
for threshold in [0.80, 0.90, 0.95, 0.98, 0.99]:
    valid_true = test_scored[test_scored["true_label"] == "valid"]
    if len(valid_true):
        false_blocks = valid_true[(valid_true["pred_label"] != "valid") & (valid_true["confidence"] >= threshold)]
        rate = len(false_blocks) / len(valid_true)
        valid_false_block_rates[str(threshold)] = {"false_blocks": int(len(false_blocks)), "valid_rows": int(len(valid_true)), "rate": float(rate)}
        print(f"valid-call false block rate @ {threshold:.2f}: {len(false_blocks)}/{len(valid_true)} = {rate:.4f}")

(DATA_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
calibration_scored.to_json(DATA_DIR / "calibration_scored.jsonl", orient="records", lines=True, force_ascii=False)
valid_scored.to_json(DATA_DIR / "validation_scored.jsonl", orient="records", lines=True, force_ascii=False)
test_scored.to_json(DATA_DIR / "test_scored.jsonl", orient="records", lines=True, force_ascii=False)

combined_training_metrics = {
    "training_run_summary": training_run_summary if "training_run_summary" in globals() else None,
    "test_metrics": test_metrics,
    "valid_false_block_rates": valid_false_block_rates,
}
(DATA_DIR / "training_metrics.json").write_text(json.dumps(combined_training_metrics, indent=2))

if "training_run_summary" in globals():
    training_run_summary["test_metrics"] = test_metrics
    training_run_summary["valid_false_block_rates"] = valid_false_block_rates
    (DATA_DIR / "training_run_summary.json").write_text(json.dumps(training_run_summary, indent=2))


## 9. Threshold calibration

Use validation data to create conservative per-label thresholds. The exported default mode remains `shadow`; enforcement should be enabled only after repo-specific eval proof.

In [ ]:
BLOCKING_LABELS = [label for label in LABELS if label not in ("valid", "deterministic_invalid")]

def expected_calibration_error(probs: np.ndarray, labels_arr: np.ndarray, n_bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = predictions == labels_arr
    ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins, endpoint=False), np.linspace(1 / n_bins, 1, n_bins)):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.any():
            ece += mask.mean() * abs(float(accuracies[mask].mean()) - float(confidences[mask].mean()))
    return float(ece)

def fit_temperature(logits: np.ndarray, labels_arr: np.ndarray) -> float:
    logits_t = torch.tensor(logits, dtype=torch.float32)
    labels_t = torch.tensor(labels_arr, dtype=torch.long)
    temperature = torch.nn.Parameter(torch.ones(()))
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=50)
    loss_fn = nn.CrossEntropyLoss()
    def closure():
        optimizer.zero_grad()
        scaled = logits_t / temperature.clamp_min(0.05)
        loss = loss_fn(scaled, labels_t)
        loss.backward()
        return loss
    optimizer.step(closure)
    return float(temperature.detach().clamp(0.05, 10.0).item())

calibration_logits = split_logits.get("calibration")
calibration_labels = split_label_ids.get("calibration")
if calibration_logits is not None and len(calibration_logits):
    TEMPERATURE = fit_temperature(calibration_logits, calibration_labels)
else:
    TEMPERATURE = 1.0
print("Temperature:", TEMPERATURE)

def add_calibrated_probs(scored: pd.DataFrame, split_name: str) -> pd.DataFrame:
    logits = split_logits[split_name]
    labels_arr = split_label_ids[split_name]
    uncal = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    cal = torch.softmax(torch.tensor(logits / TEMPERATURE), dim=-1).numpy()
    scored = scored.copy()
    scored["uncalibrated_ece"] = expected_calibration_error(uncal, labels_arr)
    scored["calibrated_ece"] = expected_calibration_error(cal, labels_arr)
    scored["calibrated_confidence"] = cal.max(axis=-1)
    scored["calibrated_pred_label"] = [id2label[int(i)] for i in cal.argmax(axis=-1)]
    for idx, label in id2label.items():
        scored[f"cal_prob_{label}"] = cal[:, int(idx)]
    return scored

calibration_scored = add_calibrated_probs(calibration_scored, "calibration")
valid_scored = add_calibrated_probs(valid_scored, "validation")
test_scored = add_calibrated_probs(test_scored, "test")

reliability_rows = []
for label in LABELS:
    col = f"cal_prob_{label}"
    if col not in valid_scored:
        continue
    for lo, hi in zip(np.linspace(0, 1, 10, endpoint=False), np.linspace(0.1, 1, 10)):
        mask = (valid_scored[col] > lo) & (valid_scored[col] <= hi)
        if mask.any():
            reliability_rows.append({
                "label": label,
                "bin_low": float(lo),
                "bin_high": float(hi),
                "rows": int(mask.sum()),
                "avg_confidence": float(valid_scored.loc[mask, col].mean()),
                "precision": float((valid_scored.loc[mask, "true_label"] == label).mean()),
            })
reliability = pd.DataFrame(reliability_rows)
reliability.to_json(DATA_DIR / "reliability_curves.jsonl", orient="records", lines=True)

def choose_threshold_for_label(scored: pd.DataFrame, label: str, target_precision: float, min_threshold: float, max_threshold: float = 0.995) -> float:
    col = f"cal_prob_{label}"
    if col not in scored:
        return 1.01
    candidates = np.linspace(min_threshold, max_threshold, 80)
    best = 1.01
    for th in candidates:
        selected = scored[scored[col] >= th]
        if selected.empty:
            continue
        precision = (selected["true_label"] == label).mean()
        valid_false_blocks = selected[(selected["true_label"] == "valid") & (label != "valid")]
        if precision >= target_precision and len(valid_false_blocks) == 0:
            best = float(th)
            break
    return best

thresholds = {
    "schema_version": "toolcall-verifier-thresholds/v1",
    "mode": "shadow",
    "default_action": "allow",
    "temperature": TEMPERATURE,
    "notes": [
        "Deterministic guardrails remain authoritative.",
        "Use ML in shadow mode first, then advisory nudges, then high-confidence enforcement only after eval proof.",
        "deterministic_invalid is never enforced by ML in this default config.",
        "wrong_tool_semantic stays conservative because current Forge telemetry showed high-confidence false positives on valid terminal/summarize calls.",
    ],
    "labels": {},
}

for label in LABELS:
    if label == "valid":
        thresholds["labels"][label] = {"action": "allow", "advisory_min_confidence": 0.0, "enforce_min_confidence": 1.01}
    elif label == "deterministic_invalid":
        thresholds["labels"][label] = {"action": "deterministic_only", "advisory_min_confidence": 1.01, "enforce_min_confidence": 1.01}
    elif label == "wrong_tool_semantic":
        thresholds["labels"][label] = {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 1.01, "enforce_min_confidence": 1.01}
    elif label == "wrong_arguments_semantic":
        thresholds["labels"][label] = {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995}
    else:
        advisory = choose_threshold_for_label(valid_scored, label, target_precision=0.90, min_threshold=0.80)
        enforce = choose_threshold_for_label(valid_scored, label, target_precision=0.98, min_threshold=0.95)
        thresholds["labels"][label] = {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": advisory, "enforce_min_confidence": enforce}

calibration_report = {
    "schema_version": "toolcall-verifier-calibration/v1",
    "temperature": TEMPERATURE,
    "calibration_ece_before": float(calibration_scored["uncalibrated_ece"].iloc[0]) if len(calibration_scored) else None,
    "calibration_ece_after": float(calibration_scored["calibrated_ece"].iloc[0]) if len(calibration_scored) else None,
    "validation_ece_before": float(valid_scored["uncalibrated_ece"].iloc[0]) if len(valid_scored) else None,
    "validation_ece_after": float(valid_scored["calibrated_ece"].iloc[0]) if len(valid_scored) else None,
    "thresholds": thresholds,
}
(DATA_DIR / "thresholds.json").write_text(json.dumps(thresholds, indent=2))
(DATA_DIR / "calibration_report.json").write_text(json.dumps(calibration_report, indent=2))
print(json.dumps(thresholds, indent=2))


## 10. Save final model and run quick inference

In [ ]:
#@title Save model and tokenizer
final_model_dir = MODEL_DIR / "final"
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

# Copy schema/metadata alongside the model for Hub upload.
for src_name in ["labels.json", "input_schema.json", "thresholds.json", "serializer_fixture.json", "training_metrics.json", "training_run_summary.json", "test_metrics.json", "calibration_report.json", "reliability_curves.jsonl", "input_schema_v1.json", "input_schema_v2.json", "serializer_fixture_v2.json", "final_response_input_schema.json"]:
    src = DATA_DIR / src_name
    if src.exists():
        shutil.copy(src, final_model_dir / src_name)

print("Saved to:", final_model_dir)
print(sorted(p.name for p in final_model_dir.iterdir()))


In [ ]:
#@title Quick inference helper
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=str(final_model_dir),
    tokenizer=str(final_model_dir),
    device=0 if torch.cuda.is_available() else -1,
    top_k=None,
)

def score_candidate(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps=None,
    completed_steps=None,
    pending_steps=None,
    terminal_tools=None,
    recent_errors=None,
):
    text = serialize_state(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
        recent_errors=recent_errors,
    )
    start = time.perf_counter()
    scores = clf(text, truncation=True, max_length=MAX_LENGTH)[0]
    elapsed_ms = (time.perf_counter() - start) * 1000
    scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    return scores[:5], elapsed_ms, text

sample = FORGE_WORKFLOWS[0]
scores, elapsed_ms, text = score_candidate(
    user_request=sample["user_request"],
    tools=sample["tools"],
    candidate={"name": "report", "arguments": {"summary": "Done."}},
    required_steps=sample["required_steps"],
    completed_steps=[],
    pending_steps=sample["required_steps"],
    terminal_tools=sample["terminal_tools"],
)
print("Latency ms:", round(elapsed_ms, 2))
print(scores)
print(text[:2000])

## 11. ONNX export, quantization, manifest, and parity checks

The Rust side should load:

- `model.onnx` or `model_quantized.onnx`
- tokenizer files
- `labels.json`
- `thresholds.json`
- `artifact_manifest.json`
- `input_schema.json`
- `serializer_fixture.json`

In [ ]:
#@title Export to ONNX with Optimum
!python -m optimum.exporters.onnx \
  --model "{final_model_dir}" \
  --task text-classification \
  "{ONNX_DIR}"

# Normalize ONNX filename for Rust config.
onnx_files = sorted(ONNX_DIR.glob("*.onnx"))
if not onnx_files:
    raise RuntimeError("No ONNX file exported")
main_onnx = ONNX_DIR / "model.onnx"
if onnx_files[0].name != "model.onnx":
    shutil.copy(onnx_files[0], main_onnx)

# Copy deployment metadata.
for src_name in ["labels.json", "input_schema.json", "thresholds.json", "serializer_fixture.json", "training_metrics.json", "training_run_summary.json", "test_metrics.json"]:
    src = DATA_DIR / src_name
    if src.exists():
        shutil.copy(src, ONNX_DIR / src_name)

# Copy tokenizer/config files from final_model_dir if Optimum did not copy all of them.
for p in final_model_dir.iterdir():
    if p.name in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "spm.model", "config.json"]:
        shutil.copy(p, ONNX_DIR / p.name)

print("ONNX artifacts:")
print(sorted(p.name for p in ONNX_DIR.iterdir()))


In [ ]:
#@title Optional dynamic quantization
from onnxruntime.quantization import quantize_dynamic, QuantType

src = ONNX_DIR / "model.onnx"
dst = ONNX_DIR / "model_quantized.onnx"
if src.exists():
    quantize_dynamic(str(src), str(dst), weight_type=QuantType.QInt8)
    print("Quantized:", dst)
else:
    print("No model.onnx found.")

In [ ]:
#@title Build artifact manifest
artifact_manifest = {
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "model_kind": "text-classification-cross-encoder",
    "base_model": BASE_MODEL,
    "label_mode": LABEL_MODE,
    "input_schema_version": INPUT_SCHEMA_VERSION,
    "serializer": SERIALIZER_VERSION,
    "max_length": MAX_LENGTH,
    "requested_gpu_profile": GPU_PROFILE,
    "run_profile": RUN_PROFILE,
    "profile_config": CFG,
    "labels_file": "labels.json",
    "thresholds_file": "thresholds.json",
    "input_schema_file": "input_schema.json",
    "serializer_fixture_file": "serializer_fixture.json",
    "training_run_summary_file": "training_run_summary.json",
    "test_metrics_file": "test_metrics.json",
    "onnx_file": "model.onnx",
    "quantized_onnx_file": "model_quantized.onnx" if (ONNX_DIR / "model_quantized.onnx").exists() else None,
    "tokenizer_required_files": [
        "tokenizer_config.json",
        "special_tokens_map.json",
        "spm.model",
    ],
    "labels": LABELS,
    "created_unix": int(time.time()),
    "precision_flags": {"fp16": USE_FP16, "bf16": USE_BF16, "tf32": USE_TF32},
    "best_model_checkpoint": getattr(trainer.state, "best_model_checkpoint", None),
    "best_metric": getattr(trainer.state, "best_metric", None),
    "best_metric_name": training_args.metric_for_best_model,
    "t4_proven_baseline": T4_PROVEN_BASELINE,
    "deployment_default": "shadow",
    "shadow_first_reason": "Experimental verifier artifacts require eval replay before advisory or enforcement promotion.",
    "supports_legacy_five_labels": False,
}
if "training_run_summary" in globals():
    artifact_manifest["training_run_summary"] = training_run_summary
(ONNX_DIR / "artifact_manifest.json").write_text(json.dumps(artifact_manifest, indent=2))
(final_model_dir / "artifact_manifest.json").write_text(json.dumps(artifact_manifest, indent=2))
print(json.dumps(artifact_manifest, indent=2))


In [ ]:
#@title PyTorch vs ONNX parity and quantization drift checks
import onnxruntime as ort

ort_session = ort.InferenceSession(str(ONNX_DIR / "model.onnx"), providers=["CPUExecutionProvider"])
pt_model = AutoModelForSequenceClassification.from_pretrained(str(final_model_dir)).eval()
pt_tokenizer = AutoTokenizer.from_pretrained(str(final_model_dir), use_fast=False)

fixture = json.loads((DATA_DIR / "serializer_fixture.json").read_text())
probe_text = fixture["serialized"]

def logits_from_pt(texts: List[str]) -> np.ndarray:
    inputs = pt_tokenizer(texts, return_tensors="pt", truncation=True, max_length=MAX_LENGTH, padding=True)
    with torch.no_grad():
        return pt_model(**inputs).logits.detach().cpu().numpy()

def logits_from_onnx(session, texts: List[str]) -> np.ndarray:
    inputs = pt_tokenizer(texts, return_tensors="pt", truncation=True, max_length=MAX_LENGTH, padding=True)
    ort_inputs = {}
    input_names = {i.name for i in session.get_inputs()}
    for k, v in inputs.items():
        if k in input_names:
            ort_inputs[k] = v.cpu().numpy()
    return session.run(None, ort_inputs)[0]

pt_logits = logits_from_pt([probe_text])
ort_logits = logits_from_onnx(ort_session, [probe_text])
pt_top = int(pt_logits.argmax(axis=-1)[0])
ort_top = int(ort_logits.argmax(axis=-1)[0])
max_abs_diff = float(np.max(np.abs(pt_logits - ort_logits)))
print("pt_top:", pt_top, id2label[pt_top])
print("ort_top:", ort_top, id2label[ort_top])
print("max_abs_diff:", max_abs_diff)
assert pt_top == ort_top, "ONNX top label does not match PyTorch top label"

parity_texts = test_scored["input_text"].sample(n=min(256, len(test_scored)), random_state=SEED).tolist()
if probe_text not in parity_texts:
    parity_texts = [probe_text] + parity_texts
pt_batch = logits_from_pt(parity_texts)
fp32_batch = logits_from_onnx(ort_session, parity_texts)
pt_top_batch = pt_batch.argmax(axis=-1)
fp32_top_batch = fp32_batch.argmax(axis=-1)

onnx_parity_report = {
    "schema_version": "toolcall-verifier-onnx-parity/v1",
    "rows": len(parity_texts),
    "pt_fp32_top_label_agreement": float((pt_top_batch == fp32_top_batch).mean()),
    "pt_fp32_max_abs_diff": float(np.max(np.abs(pt_batch - fp32_batch))),
    "quantized_present": False,
}

q_path = ONNX_DIR / "model_quantized.onnx"
if q_path.exists():
    q_sess = ort.InferenceSession(str(q_path), providers=["CPUExecutionProvider"])
    q_logits = logits_from_onnx(q_sess, parity_texts)
    q_top = q_logits.argmax(axis=-1)
    fp32_q_disagreements = int((fp32_top_batch != q_top).sum())
    onnx_parity_report.update({
        "quantized_present": True,
        "fp32_quantized_top_label_agreement": float((fp32_top_batch == q_top).mean()),
        "fp32_quantized_disagreements": fp32_q_disagreements,
        "fp32_quantized_max_abs_diff": float(np.max(np.abs(fp32_batch - q_logits))),
    })
    print("quantized disagreements:", fp32_q_disagreements, "/", len(parity_texts))

(ONNX_DIR / "onnx_parity_report.json").write_text(json.dumps(onnx_parity_report, indent=2))
(final_model_dir / "onnx_parity_report.json").write_text(json.dumps(onnx_parity_report, indent=2))
print(json.dumps(onnx_parity_report, indent=2))


## 11b. Optional final-response verifier pipeline


In [ ]:
#@title Build optional final-response verifier dataset and artifacts
FINAL_RESPONSE_MODEL_DIR = None
FINAL_RESPONSE_ONNX_DIR = None
FINAL_RESPONSE_ARTIFACT_DIR = None

if ENABLE_FINAL_RESPONSE_VERIFIER:
    def release_toolcall_training_memory():
        release_names = [
            "trainer", "model", "pt_model", "ort_session", "q_sess",
            "tokenized", "ds", "data_collator", "class_weights",
            "sample_batch", "collated", "split_logits", "split_label_ids",
            "calibration_scored", "valid_scored", "test_scored",
            "pt_logits", "ort_logits", "pt_batch", "fp32_batch", "q_logits",
            "balanced", "train_df", "valid_full_df", "valid_df", "test_df", "calibration_df",
        ]
        released = []
        for name in release_names:
            if name in globals():
                globals().pop(name, None)
                released.append(name)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
            allocated = torch.cuda.memory_allocated() / (1024 ** 3)
            reserved = torch.cuda.memory_reserved() / (1024 ** 3)
            print(f"CUDA after cleanup: allocated={allocated:.2f}GB reserved={reserved:.2f}GB")
        if released:
            print("Released prior tool-call training objects:", ", ".join(released))

    release_toolcall_training_memory()

    FR_MAX_LENGTH = FINAL_RESPONSE_MAX_LENGTH_OVERRIDE or min(MAX_LENGTH, 768)
    FR_NUM_EPOCHS = FINAL_RESPONSE_NUM_EPOCHS_OVERRIDE or NUM_EPOCHS
    FR_TRAIN_BATCH_SIZE = FINAL_RESPONSE_TRAIN_BATCH_SIZE_OVERRIDE or min(TRAIN_BATCH_SIZE, 16)
    FR_EVAL_BATCH_SIZE = FINAL_RESPONSE_EVAL_BATCH_SIZE_OVERRIDE or min(EVAL_BATCH_SIZE, 32)
    FR_GRAD_ACCUM = FINAL_RESPONSE_GRAD_ACCUM_OVERRIDE or max(
        1, math.ceil((TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) / FR_TRAIN_BATCH_SIZE)
    )
    print("Final-response memory profile:", {
        "max_length": FR_MAX_LENGTH,
        "epochs": FR_NUM_EPOCHS,
        "train_batch_size": FR_TRAIN_BATCH_SIZE,
        "eval_batch_size": FR_EVAL_BATCH_SIZE,
        "grad_accum": FR_GRAD_ACCUM,
        "max_per_label": FINAL_RESPONSE_MAX_PER_LABEL,
    })

    if not final_response_rows:
        synthetic_specs = [
            ("valid_final_response", GROUNDED_SYNTHESIS_POSITIVE, 1.0),
            ("missing_tool_fact", "Aisha Patel is recommended for the role.", 0.10),
            ("contradicts_tool_result", "Select Sarah Chen because she can start immediately.", 0.05),
            ("unsupported_claim", "Aisha Patel is recommended and has already accepted the offer.", 0.05),
            ("failed_to_acknowledge_data_gap", "The missing compensation data confirms Aisha is best.", 0.05),
        ]
        final_response_rows = []
        for repeat in range(2):
            for label, response, rank in synthetic_specs:
                final_response_rows.append(make_final_response_row(
                    "forge_final_synthetic",
                    label,
                    GROUNDED_SYNTHESIS_USER,
                    response,
                    rank,
                    required_facts=GROUNDED_SYNTHESIS_FACTS,
                    tool_trace=["get_open_role", "get_candidate_pool", "get_compatibility_check", "submit_hiring_decision"],
                    tool_results=[{"tool_name": "get_compatibility_check", "content": "Sarah has an active non-compete; James requires sponsorship; Aisha can start immediately."}],
                    metadata={"generator": "forge_final_synthetic", "scenario_family": "grounded_synthesis", "example_group_id": f"final_synth_{label}_{repeat}"},
                ))

    final_df = pd.DataFrame([asdict(r) for r in final_response_rows])
    final_df["example_group_id"] = final_df["metadata"].map(lambda m: m.get("example_group_id") if isinstance(m, dict) else None)
    final_df["input_text"] = final_df["input"].map(serialize_final_response_state_v1)
    final_df["labels"] = final_df["label"].map(final_response_label2id)
    if final_df["labels"].isna().any():
        bad = sorted(final_df.loc[final_df["labels"].isna(), "label"].unique())
        raise ValueError(f"Unknown final-response labels: {bad}")
    final_df["labels"] = final_df["labels"].astype(int)

    final_pieces = []
    for label, group in final_df.groupby("label", sort=False):
        final_pieces.append(group.sample(n=min(len(group), FINAL_RESPONSE_MAX_PER_LABEL), random_state=SEED))
    final_balanced = pd.concat(final_pieces, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    fr_train_df, fr_valid_df, fr_test_df = group_split(final_balanced)
    for name, split_df in [("final_response_train", fr_train_df), ("final_response_validation", fr_valid_df), ("final_response_test", fr_test_df)]:
        path = DATA_DIR / f"{name}.jsonl"
        with path.open("w") as f:
            for row in split_df.to_dict(orient="records"):
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print("Final-response split", name, len(split_df), "groups", split_df["example_group_id"].nunique(), path)

    final_dataset_path = DATA_DIR / "final_response_verifier_dataset.jsonl"
    with final_dataset_path.open("w") as f:
        for row in final_balanced.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    fr_ds = DatasetDict({
        "train": Dataset.from_pandas(fr_train_df.reset_index(drop=True), preserve_index=False),
        "validation": Dataset.from_pandas(fr_valid_df.reset_index(drop=True), preserve_index=False),
        "test": Dataset.from_pandas(fr_test_df.reset_index(drop=True), preserve_index=False),
    })
    fr_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

    def tokenize_final_response_batch(batch):
        encoded = fr_tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=FR_MAX_LENGTH,
            padding=False,
        )
        encoded["labels"] = [int(x) for x in batch["labels"]]
        return encoded

    fr_tokenized = fr_ds.map(
        tokenize_final_response_batch,
        batched=True,
        remove_columns=fr_ds["train"].column_names,
        desc="Tokenizing final-response verifier rows",
    )
    fr_tokenized = fr_tokenized.cast_column("labels", Value("int64"))
    fr_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    fr_data_collator = DataCollatorWithPadding(tokenizer=fr_tokenizer)

    fr_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=len(FINAL_RESPONSE_LABELS),
        id2label={int(k): v for k, v in final_response_id2label.items()},
        label2id=final_response_label2id,
    )
    if fr_tokenizer.pad_token_id is not None:
        fr_model.config.pad_token_id = fr_tokenizer.pad_token_id
    if fr_tokenizer.eos_token_id is not None:
        fr_model.config.eos_token_id = fr_tokenizer.eos_token_id
    if fr_tokenizer.bos_token_id is not None:
        fr_model.config.bos_token_id = fr_tokenizer.bos_token_id

    FR_ALL_LABEL_IDS = list(range(len(FINAL_RESPONSE_LABELS)))

    def compute_final_response_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        present_precision, present_recall, present_f1, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
        full_precision, full_recall, full_f1, _ = precision_recall_fscore_support(
            labels, preds, labels=FR_ALL_LABEL_IDS, average="macro", zero_division=0
        )
        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_precision": present_precision,
            "macro_recall": present_recall,
            "macro_f1": present_f1,
            "macro_precision_all_labels": full_precision,
            "macro_recall_all_labels": full_recall,
            "macro_f1_all_labels": full_f1,
        }

    def build_final_response_class_weights(train_frame: pd.DataFrame) -> Optional[torch.Tensor]:
        if not USE_CLASS_WEIGHTS:
            return None
        counts = train_frame["label"].value_counts()
        weights = []
        for label in FINAL_RESPONSE_LABELS:
            count = counts.get(label, 0)
            if count <= 0:
                weights.append(0.0)
            else:
                raw = len(train_frame) / (len(FINAL_RESPONSE_LABELS) * count)
                weights.append(min(float(raw), MAX_CLASS_WEIGHT))
        weights = np.array(weights, dtype=np.float32)
        nonzero = weights[weights > 0]
        if len(nonzero):
            weights[weights > 0] = weights[weights > 0] / nonzero.mean()
        return torch.tensor(weights, dtype=torch.float)

    fr_model_dir = MODEL_DIR / "final_response"
    fr_training_args = TrainingArguments(
        output_dir=str(fr_model_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=FR_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=FR_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=FR_GRAD_ACCUM,
        num_train_epochs=FR_NUM_EPOCHS,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=USE_FP16,
        bf16=USE_BF16,
        tf32=USE_TF32,
        optim=OPTIMIZER,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
        dataloader_num_workers=DATALOADER_NUM_WORKERS if USE_CUDA else 0,
        dataloader_pin_memory=USE_CUDA,
        group_by_length=GROUP_BY_LENGTH,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
    )

    fr_trainer = WeightedTrainer(
        class_weights=build_final_response_class_weights(fr_train_df),
        model=fr_model,
        args=fr_training_args,
        train_dataset=fr_tokenized["train"],
        eval_dataset=fr_tokenized["validation"],
        processing_class=fr_tokenizer,
        data_collator=fr_data_collator,
        compute_metrics=compute_final_response_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    fr_train_result = fr_trainer.train()
    fr_trainer.save_state()
    fr_test_metrics = fr_trainer.evaluate(fr_tokenized["test"])

    fr_final_model_dir = fr_model_dir / "final"
    fr_trainer.save_model(str(fr_final_model_dir))
    fr_tokenizer.save_pretrained(str(fr_final_model_dir))

    final_labels_doc = {
        "label_mode": "production",
        "labels": FINAL_RESPONSE_LABELS,
        "label2id": final_response_label2id,
        "id2label": final_response_id2label,
    }
    final_thresholds = {
        "schema_version": FINAL_RESPONSE_THRESHOLDS_SCHEMA_VERSION,
        "mode": "shadow",
        "default_action": "allow",
        "labels": {
            "valid_final_response": {"action": "allow", "advisory_min_confidence": 0.0, "enforce_min_confidence": 1.01},
            "missing_tool_fact": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
            "contradicts_tool_result": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
            "unsupported_claim": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
            "failed_to_acknowledge_data_gap": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
        },
    }
    final_manifest = {
        "artifact_schema_version": FINAL_RESPONSE_ARTIFACT_SCHEMA_VERSION,
        "model_kind": "text-classification-cross-encoder",
        "base_model": BASE_MODEL,
        "label_mode": "production",
        "input_schema_version": FINAL_RESPONSE_INPUT_SCHEMA_VERSION,
        "serializer": FINAL_RESPONSE_SERIALIZER_VERSION,
        "max_length": FR_MAX_LENGTH,
        "onnx_file": "model.onnx",
        "quantized_onnx_file": "model_quantized.onnx",
        "labels": FINAL_RESPONSE_LABELS,
        "deployment_default": "shadow",
        "shadow_first_reason": "experimental final-response verifier; promote only after eval replay",
        "created_unix": int(time.time()),
    }
    final_training_provenance = {
        "schema_version": "final-response-verifier-training-provenance/v1",
        "base_model": BASE_MODEL,
        "run_profile": RUN_PROFILE,
        "memory_profile": {
            "max_length": FR_MAX_LENGTH,
            "epochs": FR_NUM_EPOCHS,
            "train_batch_size": FR_TRAIN_BATCH_SIZE,
            "eval_batch_size": FR_EVAL_BATCH_SIZE,
            "grad_accum": FR_GRAD_ACCUM,
            "max_per_label": FINAL_RESPONSE_MAX_PER_LABEL,
        },
        "rows": int(len(final_balanced)),
        "train_rows": int(len(fr_train_df)),
        "validation_rows": int(len(fr_valid_df)),
        "test_rows": int(len(fr_test_df)),
        "train_metrics": fr_train_result.metrics,
        "test_metrics": fr_test_metrics,
    }

    for target in [fr_final_model_dir]:
        (target / "labels.json").write_text(json.dumps(final_labels_doc, indent=2))
        (target / "input_schema.json").write_text(json.dumps(FINAL_RESPONSE_INPUT_SCHEMA, indent=2))
        (target / "thresholds.json").write_text(json.dumps(final_thresholds, indent=2))
        (target / "artifact_manifest.json").write_text(json.dumps(final_manifest, indent=2))
        (target / "training_provenance.json").write_text(json.dumps(final_training_provenance, indent=2))

    fr_onnx_dir = WORKDIR / "final_response_onnx"
    fr_onnx_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "python", "-m", "optimum.exporters.onnx",
        "--model", str(fr_final_model_dir),
        "--task", "text-classification",
        str(fr_onnx_dir),
    ], check=True)

    fr_onnx_files = sorted(fr_onnx_dir.glob("*.onnx"))
    if not fr_onnx_files:
        raise RuntimeError("No final-response ONNX file exported")
    fr_main_onnx = fr_onnx_dir / "model.onnx"
    if fr_onnx_files[0].name != "model.onnx":
        shutil.copy(fr_onnx_files[0], fr_main_onnx)

    from onnxruntime.quantization import quantize_dynamic, QuantType
    if fr_main_onnx.exists():
        quantize_dynamic(str(fr_main_onnx), str(fr_onnx_dir / "model_quantized.onnx"), weight_type=QuantType.QInt8)

    for p in fr_final_model_dir.iterdir():
        if p.name in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "spm.model", "config.json", "labels.json", "input_schema.json", "thresholds.json", "artifact_manifest.json", "training_provenance.json"]:
            shutil.copy(p, fr_onnx_dir / p.name)

    FINAL_RESPONSE_MODEL_DIR = fr_final_model_dir
    FINAL_RESPONSE_ONNX_DIR = fr_onnx_dir
    FINAL_RESPONSE_ARTIFACT_DIR = fr_onnx_dir
    print("Final-response dataset:", final_dataset_path, "rows", len(final_balanced))
    print("Final-response model:", FINAL_RESPONSE_MODEL_DIR)
    print("Final-response ONNX artifact:", FINAL_RESPONSE_ONNX_DIR)
else:
    print("Final-response verifier pipeline skipped. Set ENABLE_FINAL_RESPONSE_VERIFIER=True in Colab to train/export artifacts.")


## 12. Package and upload artifacts

Upload is enabled by default and private by default. Keep it private until public dataset licenses and any proprietary Forge traces are verified.

In [ ]:
#@title Zip dataset, model, and ONNX artifacts
zip_path = shutil.make_archive(
    base_name=str(ARTIFACT_DIR / "toolcall_verifier_artifacts"),
    format="zip",
    root_dir=str(WORKDIR),
    base_dir=".",
)
print("Wrote:", zip_path)

In [ ]:
#@title Upload dataset/model/ONNX artifacts to Hugging Face Hub
if UPLOAD_TO_HUB:
    from huggingface_hub import HfApi, create_repo, upload_folder, upload_file
    api = HfApi()
    create_repo(HF_DATASET_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True)
    upload_folder(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        folder_path=str(DATA_DIR),
        commit_message="Add verifier datasets and schemas",
    )
    create_repo(HF_MODEL_REPO, repo_type="model", private=PRIVATE, exist_ok=True)
    upload_folder(
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        folder_path=str(final_model_dir),
        path_in_repo="hf_model",
        commit_message="Add fine-tuned classifier checkpoint",
    )
    upload_folder(
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        folder_path=str(ONNX_DIR),
        path_in_repo="onnx",
        commit_message="Add ONNX deployment artifacts",
    )
    upload_file(
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        path_or_fileobj=zip_path,
        path_in_repo="artifacts/toolcall_verifier_artifacts.zip",
        commit_message="Add packaged tool-call verifier artifacts",
    )
    if ENABLE_FINAL_RESPONSE_VERIFIER and FINAL_RESPONSE_MODEL_DIR and FINAL_RESPONSE_ONNX_DIR:
        create_repo(HF_FINAL_RESPONSE_MODEL_REPO, repo_type="model", private=PRIVATE, exist_ok=True)
        upload_folder(
            repo_id=HF_FINAL_RESPONSE_MODEL_REPO,
            repo_type="model",
            folder_path=str(FINAL_RESPONSE_MODEL_DIR),
            path_in_repo="hf_model",
            commit_message="Add final-response verifier checkpoint",
        )
        upload_folder(
            repo_id=HF_FINAL_RESPONSE_MODEL_REPO,
            repo_type="model",
            folder_path=str(FINAL_RESPONSE_ONNX_DIR),
            path_in_repo="onnx",
            commit_message="Add final-response verifier ONNX artifacts",
        )
    print("Uploaded dataset and model artifacts.")
else:
    print("Upload disabled.")


## 13. Recommended ablation matrix

The first completed T4 baseline is good enough to treat as the reference run:

```text
Epoch 4 validation accuracy:       0.839146
Epoch 4 validation macro_f1:       0.741871
Epoch 4 macro_f1_all_labels:       0.593497
Runtime on T4:                     about 1h25m
```

Run these in separate Colab sessions or with separate output directories. The notebook is Colab-first; do not expect local execution to reproduce Colab GPU/runtime behavior. Upload remains enabled by default, with artifacts marked shadow-first until eval replay proves safety.

| Run | Profile | Label mode | Class weights | Forge augmentation | Purpose |
|---|---|---|---:|---:|---|
| smoke/export check | `debug_smoke` | `production` | off | off | validate Colab cells, schemas, ONNX, upload |
| fast T4 iteration | `t4_fast` | `production` | off | off | quick quality check on cheaper GPU |
| proven T4 baseline | `t4_proven` | `production` | off | off | current reference artifact candidate |
| L4/A100 long context | `l4_balanced` or `a100_40gb` | `production` | off | off | test p95 token retention |
| 80-100GB fast pass | `high_vram_fast` | `production` | off | off | first run on the 98GB GPU |
| 80-100GB quality | `high_vram_quality` | `production` | off | off | wider data + 1280-token context |
| 80-100GB full context | `high_vram_full_context` | `production` | off | off | expensive 1536-token context ablation |
| diagnostic debug | `t4_proven` | `diagnostic` | off | off | find failure categories |
| Forge contract | `t4_proven` | `diagnostic` | off | on | test contract-specific labels |
| weighted minority | `a100_40gb` or higher | `production` | on | off | only if minority recall is poor |

For your current T4 timing, use `GPU_PROFILE="t4_proven"` or leave `GPU_PROFILE="auto"` on a T4 runtime. For the 98GB runtime, start with `GPU_PROFILE="high_vram_fast"`, then run `GPU_PROFILE="high_vram_quality"` if the fast pass is healthy.

The completed T4 run improved through epoch 4, so the notebook keeps 4 epochs for the T4 reference profile and 5 epochs for high-VRAM quality profiles. Early stopping remains enabled, and the saved model is selected by validation `macro_f1`, not by the final epoch blindly.


Recommended eval replay after upload: `no_classifier`, `classifier_fp32_onnx_shadow`, `classifier_quantized_onnx_shadow`, `classifier_fp32_onnx_advisory`, and `classifier_quantized_onnx_advisory`. Promote beyond shadow only if valid-call false objections stay at zero and targeted scenario families improve.


# 14. Rust implementation appendix

## Integration rule

The classifier must run after deterministic checks, never before them.

```text
1. Parse provider response.
2. Validate format, known tool names, and JSON-schema arguments.
3. Enforce required steps, prerequisites, terminal rules, and unsafe batches.
4. If the call is still valid-looking, run the classifier.
5. Shadow mode: log classifier verdict only.
6. Advisory mode: use classifier verdict to choose better nudges.
7. Enforce mode: block only high-confidence semantic labels after eval proof.
```

Your codebase already has the right seams:

- `guardrails/guardrails.rs`: deterministic check pipeline
- `guardrails/policy.rs`: `GuardrailState` and reserved semantic-scoring violation shape
- `guardrails/history.rs`: bounded recent failure memory
- `proxy/handler.rs`: parses OpenAI tools and builds guarded workflow requests
- `bin/forge-eval/report.rs`: good place to add classifier telemetry

## Add modules

```text
guardrails/scoring_context.rs
guardrails/scorer.rs
guardrails/model_scorer.rs        # feature = "onnx-classifier"
guardrails/classifier_artifact.rs
```

## Suggested Rust API

```rust
use crate::clients::base::ToolCall;
use crate::core::tool_spec::ToolSpec;
use crate::guardrails::policy::GuardrailState;

#[derive(Debug, Clone, serde::Serialize, serde::Deserialize)]
pub struct ScoringContext {
    pub schema_version: String,
    pub user_request: String,
    pub required_steps: Vec<String>,
    pub completed_steps: Vec<String>,
    pub pending_steps: Vec<String>,
    pub terminal_tools: Vec<String>,
    pub available_tools: Vec<ToolSpec>,
    pub recent_errors: Vec<String>,
    pub deterministic_state: GuardrailState,
}

#[derive(Debug, Clone, serde::Serialize, serde::Deserialize)]
pub enum ToolCallClass {
    Valid,
    WrongToolSemantic,
    ToolNotNeeded,
    NeedsClarification,
    DeterministicInvalid,
}

#[derive(Debug, Clone)]
pub enum ClassifierAction {
    Allow,
    ShadowLog,
    AdvisoryNudge,
    EnforceBlock,
}

#[derive(Debug, Clone)]
pub struct ToolCallScore {
    pub label: ToolCallClass,
    pub confidence: f32,
    pub rank_score: f32,
    pub action: ClassifierAction,
    pub latency_ms: f32,
}

pub trait ToolCallScorer: Send + Sync {
    fn score(&self, ctx: &ScoringContext, candidate: &ToolCall) -> anyhow::Result<ToolCallScore>;
}
```

## Artifact loader

`classifier_artifact.rs` should validate:

```text
artifact_manifest.json exists and includes training_run_summary/test_metrics provenance
artifact_schema_version == "toolcall-verifier-artifact/v1"
input_schema_version == "toolcall-verifier-input/v1"
serializer == "serialize_state_v1"
labels.json labels match model config
thresholds.json has every deployed label
tokenizer files exist
ONNX file exists
```

Fail closed for loading errors, but fail open for scoring errors in `shadow` and `advisory` mode.

## Serializer parity test

Use `serializer_fixture.json` from this notebook. Add a Rust unit test that constructs the same `ScoringContext` and `ToolCall`, serializes it, and byte-compares against the fixture's `serialized` string. This catches silent train/inference drift.

## CLI/config flags

Add these to the proxy binary:

```text
--classifier-dir <path>
--classifier-mode off|shadow|advisory|enforce
--classifier-max-latency-ms <n>
FORGE_CLASSIFIER_DIR
FORGE_CLASSIFIER_MODE
FORGE_CLASSIFIER_MAX_LATENCY_MS
```

Default should be `off` unless `--classifier-dir` is passed. First production rollout should use `shadow`.

## Cargo feature sketch

```toml
[features]
default = []
onnx-classifier = ["dep:ort", "dep:tokenizers"]

[dependencies]
ort = { version = "2", optional = true }
tokenizers = { version = "0.20", optional = true }
serde = { version = "1", features = ["derive"] }
serde_json = "1"
anyhow = "1"
```

Pin exact `ort` and `tokenizers` versions after local compile. The notebook writes `spm.model`; if you want Rust inference through `tokenizers`, verify whether a `tokenizer.json` is available. If not, either export a fast tokenizer where safe or use a sidecar scorer process until tokenizer parity is solved.

## Telemetry fields for `forge-eval`

Add these to each report row:

```json
{
  "classifier_enabled": true,
  "classifier_mode": "shadow",
  "classifier_label": "wrong_tool_semantic",
  "classifier_confidence": 0.93,
  "classifier_action": "shadow_log",
  "classifier_model_version": "toolcall-verifier-production-v1",
  "classifier_latency_ms": 4.2
}
```

## Enforcement invariant

The classifier may block only semantic labels:

```text
wrong_tool_semantic
tool_not_needed
needs_clarification
```

It must not override deterministic validation, accept malformed tool calls, execute tools, or rewrite arguments.
